# 실험판

**셀을 위에서부터 차례대로 실행**하면 됨. 함수 네 개와, 그 함수들이 같이 쓰는 채점 함수 하나로 돼 있음.

| 함수 | 하는 일 | 시간 |
|---|---|---|
| `데이터불러오기()` | parquet 두 개 → 전역 변수 `df` · `코드` · `조건` | 30초 |
| `스코어뽑기()` | 저장된 모델로 확률 p 다시 계산 (평소엔 안 씀) | — |
| `빔서치()` | 집단 수천 개를 뒤져 t 가 높은 것만 | 약 1분 |
| `실험()` | 가짜 하락을 심고 빔서치가 찾아내는지 | δ 하나당 1분 |
| `집단점수()` | 집단 하나 채점 — 위 둘이 이걸 부름 | 0초 |

`df` 는 그냥 DataFrame 이라 `df.head()`, `df[df['age'] == '20s']` 처럼 평소대로 보면 됨.

## 1. 준비

In [1]:
import time

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

## 2. 데이터 불러오기

`데이터불러오기()` 가 전역 변수 네 개를 만듦.

| 변수 | 내용 |
|---|---|
| `df` | 행 하나 = 유저 × 기준일. **학습 행이 먼저, 평가 행이 나중** |
| `코드` | 축 이름 → 행마다 몇 번째 값인지 (정수). 빔서치가 `bincount` 로 합계를 낼 때 씀 |
| `이름` | 축 이름 → 그 번호가 무슨 값인지. `이름['age'][2]` 는 `'20s'` |
| `조건` | 집단을 가르는 조건 290개 |

```python
이름['age']          # ['(결측)', '10s', '20s', '30s', '40+']
코드['age'][:5]      # array([3, 3, 2, 3, 1])  ← df['age'] 가 30s, 30s, 20s, 30s, 10s
```

`df` 의 주요 열 — `구간`(학습/평가) · `day`(0 = 3/1) · `d1`(실제 재방문) · `p`(기대 확률) ·
`dev`(편차 = d1 − p, **양수면 기대보다 더 돌아온 것**) · `가중치`(유효 인원)

| 인자 | 기본값 | 바꾸면 |
|---|---|---|
| `학습마지막` | 14 | 학습 기간 마지막 기준일 (14 = 3/15) |
| `평가마지막` | **16** | 평가 기간 마지막. 16 = 3/17 (주 분석, 2일 창). 14일 창은 28 |
| `구간수` | 5 | 연속 변수 분위 구간 수 |
| `최소범주행` | 1500 | 이보다 드문 범주 값은 '기타'로 묶음 |

In [2]:
# 집단을 가를 때 쓰는 축 23개
연속축 = ['days_since_signup', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid',
        'discount', 'days_to_expire', 'act_days', 'mins_per_day', 'pct_full',
        'pct_unique', 'days_since_last', 'trend', 'weekend_ratio']
범주축 = ['city', 'age', 'gender', 'channel', 'profile', 'payment_method_id']
이진축 = ['has_tx', 'is_auto_renew', 'is_cancel', 'no_habit']
축목록 = 연속축 + 범주축 + 이진축

# 분위수로 자르면 안 되는 두 축만 EDA 에서 쓰던 구간을 그대로 씀
구간고정 = {
    'act_days':        [-1, 0, 3, 10, 20, 28],
    'days_since_last': [-1, 0, 2, 6, 13, 27]}
이름고정 = {
    'act_days':        ['0 (미측정)', '1-3일', '4-10일', '11-20일', '21-28일'],
    'days_since_last': ['0일(2/28까지)', '1-2일', '3-6일', '7-13일', '14일+']}


def 데이터불러오기(학습마지막=14, 평가마지막=16, 구간수=5, 최소범주행=1500):
    """parquet 두 개를 읽어 전역 변수 세 개를 만듦.

      df   : 행 하나 = 유저 × 기준일. 학습 행이 먼저, 평가 행이 나중에 옴
      코드 : 축 이름 → 행마다 '몇 번째 값인지' (정수, 0 = 결측)
      이름 : 축 이름 → 그 번호가 무슨 값인지.  이름['age'][2] == '20s'
      조건 : 집단을 가르는 조건 290개. [{'이름', '축', '값들'}, ...]

    인자
      학습마지막 : 학습(정상) 기간 마지막 기준일. 14 = 3/15
      평가마지막 : 평가 기간 마지막 기준일. **16 = 3/17 이 주 분석(2일 창)**.
                   보조 분석인 14일 창은 28(= 3/29). 29(= 3/30)는 경계 효과라 안 씀
      구간수     : 연속 변수를 몇 분위로 자를지
      최소범주행 : 이보다 드문 범주 값은 '기타' 로 묶음
    """
    global df, 코드, 이름, 조건
    시작 = time.time()

    mart = pd.read_parquet('data/mart.parquet', columns=['u', 'day', 'base_dow'] + 축목록)
    pred = pd.read_parquet('data/pred.parquet', columns=['u', 'day', 'd1', 'p', 'dev'])
    assert len(mart) == len(pred)
    assert (mart['u'].values == pred['u'].values).all()
    assert (mart['day'].values == pred['day'].values).all()

    df = mart
    df['d1'] = pred['d1'].values         # 다음 날 돌아왔나 (0/1)
    df['p'] = pred['p'].values           # 기대 모델이 낸 확률
    df['dev'] = pred['dev'].values       # 편차 = d1 − p.  양수면 기대보다 더 돌아온 것

    # 평가 기간 뒤쪽을 버리고, 학습 행이 앞에 오도록 다시 쌓음
    df = df[df['day'] <= 평가마지막].copy()
    df['구간'] = np.where(df['day'] <= 학습마지막, '학습', '평가')
    df = pd.concat([df[df['구간'] == '학습'], df[df['구간'] == '평가']])
    df = df.reset_index(drop=True)

    # 유효 인원 가중치 — 한 유저가 평가 기간에 여러 행을 차지하므로 행 수로 세면 부풀음
    평가 = df['구간'].values == '평가'
    행수 = df.loc[평가, 'u'].value_counts()
    df['가중치'] = 0.0
    df.loc[평가, '가중치'] = 1.0 / df.loc[평가, 'u'].map(행수).values

    # ── 축마다 값을 구간으로 나눠 코드(정수)로 바꿈 ──────────────────────
    학습 = ~평가
    코드 = {}      # 축 → 행마다 몇 번째 값인지
    이름 = {}      # 축 → 번호별 값 이름
    경계 = {}
    for 축 in 연속축:
        값 = df[축].values.astype('float64')
        if 축 in 구간고정:
            자를곳 = np.array(구간고정[축], dtype='float64')
            값이름 = list(이름고정[축])
        else:
            자를곳 = np.unique(np.nanquantile(값[학습], np.linspace(0, 1, 구간수 + 1)))
            값이름 = []
            for i in range(len(자를곳) - 1):
                if i == 0:
                    값이름.append('≤{:g}'.format(자를곳[i + 1]))
                elif i == len(자를곳) - 2:
                    값이름.append('>{:g}'.format(자를곳[i]))
                else:
                    값이름.append('{:g}~{:g}'.format(자를곳[i], 자를곳[i + 1]))
        자를곳[0] = -np.inf
        자를곳[-1] = np.inf
        코드[축] = (pd.cut(값, 자를곳).codes + 1).astype('int16')   # 결측은 0
        이름[축] = ['(결측)'] + 값이름
        경계[축] = 자를곳

    for 축 in 범주축:
        값 = df[축].astype('category')
        c =값.cat.codes.values.astype('int16') + 1
        값이름 = ['(결측)'] + [str(v) for v in 값.cat.categories]
        세기 = np.bincount(c, minlength=len(값이름))
        드문값 = [i for i in range(1, len(값이름)) if 세기[i] < 최소범주행]
        if 드문값:
            c[np.isin(c, 드문값)] = len(값이름)
            값이름 = 값이름 + ['기타({}개 값)'.format(len(드문값))]
        코드[축] = c
        이름[축] = 값이름

    for 축 in 이진축:
        값 = df[축].values
        if 값.dtype == bool:
            코드[축] = 값.astype('int16') + 1
        else:
            값 = 값.astype('float64')
            코드[축] = np.where(np.isnan(값), 0, np.where(값 > 0.5, 2, 1)).astype('int16')
        이름[축] = ['(결측)', '아니오', '예']

    # 안 쓰이는 코드 번호를 없애 번호를 0부터 촘촘하게 다시 매김
    for 축 in 코드:
        쓰인값 = np.unique(코드[축])
        새번호 = np.zeros(len(이름[축]), dtype='int16')
        for 새, 옛 in enumerate(쓰인값):
            새번호[옛] = 새
        코드[축] = 새번호[코드[축]]
        이름[축] = [이름[축][i] for i in 쓰인값]

    # ── 조건 만들기 ──────────────────────────────────────────────────────
    조건 = []
    for 축 in 연속축:
        값번호 = [i for i in range(len(이름[축])) if 이름[축][i] != '(결측)']
        for i in 값번호:                                    # 구간 하나
            조건.append({'이름': 축 + ' ' + 이름[축][i], '축': 축, '값들': [i]})
        # 이하 / 초과. 양 끝은 **구간 하나짜리와 같은 집단**이 되는 경우가 있어 빼야 함
        # (그대로 두면 같은 집단이 빔 20칸을 두 자리 차지함 — 10/11 발견).
        # ⚠️ 단 **이름이 실제로 겹칠 때만** 뺄 것. 분위수로 이름을 자동 생성하는 축은
        #    첫 구간이 '≤a' 끝 구간이 '>d' 라 겹치지만, `이름고정` 을 쓰는
        #    act_days·days_since_last 는 '0 (미측정)'·'21-28일' 이라 **안 겹침**.
        #    10/11 에 이걸 안 보고 무조건 뺐다가 멀쩡한 조건 4개가 같이 사라졌었음
        #    (act_days ≤0·>20, days_since_last ≤0·>13 — 앞의 둘은 평가의 52%·69%)
        이미쓴이름 = {c['이름'] for c in 조건 if c['축'] == 축}
        for j in range(len(값번호) - 1):
            자른값 = 경계[축][j + 1]
            작은쪽 = '{} ≤{:g}'.format(축, 자른값)
            큰쪽 = '{} >{:g}'.format(축, 자른값)
            if 작은쪽 not in 이미쓴이름:
                조건.append({'이름': 작은쪽, '축': 축, '값들': 값번호[:j + 1]})
            if 큰쪽 not in 이미쓴이름:
                조건.append({'이름': 큰쪽, '축': 축, '값들': 값번호[j + 1:]})
        if 이름[축][0] == '(결측)':
            조건.append({'이름': 축 + ' 결측', '축': 축, '값들': [0]})
    for 축 in 범주축 + 이진축:
        전부 = list(range(len(이름[축])))
        for i in 전부:
            조건.append({'이름': '{} = {}'.format(축, 이름[축][i]), '축': 축, '값들': [i]})
            if len(전부) > 2:                                # 값이 2개면 ≠ 는 = 와 같음
                조건.append({'이름': '{} ≠ {}'.format(축, 이름[축][i]), '축': 축,
                           '값들': [j for j in 전부 if j != i]})

    for c in 조건:                                           # 조건마다 고를 값 표시를 미리 만들어 둠
        표시 = np.zeros(len(이름[c['축']]), dtype=bool)
        표시[c['값들']] = True
        c['표시'] = 표시

    print('df {:,}행 (학습 {:,} + 평가 {:,})'.format(len(df), (~평가).sum(), 평가.sum()))
    print('유효 인원 {:,.0f}명 / 축 {}개 / 조건 {}개'.format(
        df['가중치'].sum(), len(코드), len(조건)))
    print('학습 평균 편차 {:+.4f}%p / 평가 평균 편차 {:+.4f}%p   ({:.0f}초)'.format(
        df.loc[~평가, 'dev'].mean() * 100, df.loc[평가, 'dev'].mean() * 100, time.time() - 시작))

# ── 주 분석 설정 (10/11) ─────────────────────────────────────────────
# 평가 창은 2일이 주 분석, 14일은 보조. 아래 한 줄만 바꿔 이 셀을 다시 실행하면
# 8절·9절이 그 창으로 돌아가고, 결과 파일도 창 이름으로 따로 저장됨.
#   2일  → '2일',  16, 1400      (평가 기준일 3/16~3/17)
#   14일 → '14일', 28, 10000     (평가 기준일 3/16~3/29)
창이름, 평가끝, 보정행 = '2일', 16, 1400


def 저장경로(이름):
    """창마다 파일을 따로 남겨 두 분석이 서로 덮어쓰지 않게 함.
       지금 df 의 평가 기간이 창이름과 맞는지 확인함 — 안 맞으면 엉뚱한 파일에 덮어쓰게 됨"""
    실제끝 = int(df.loc[df['구간'] == '평가', 'day'].max())
    assert 실제끝 == 평가끝, (
        '창이름 {!r} 인데 지금 df 의 평가는 day {} 까지임 (창이름에 맞으려면 {}). '
        '이 셀을 다시 실행할 것'.format(창이름, 실제끝, 평가끝))
    return 'results/{}_{}.csv'.format(이름, 창이름)


데이터불러오기(학습마지막=14, 평가마지막=평가끝)
print('창 {} / 최소보정행 {:,}'.format(창이름, 보정행))

df 10,007,752행 (학습 8,802,984 + 평가 1,204,768)
유효 인원 728,502명 / 축 23개 / 조건 268개
학습 평균 편차 -0.0000%p / 평가 평균 편차 +1.0819%p   (17초)
창 2일 / 최소보정행 1,400


## 3. 모델로 스코어 뽑기

**평소엔 안 돌려도 됨.** p 와 편차가 이미 `df` 에 들어 있음. 변수나 모델을 바꿨을 때만 씀.

```python
p = 스코어뽑기(행번호=np.arange(1000))     # 앞 1000행만
```

In [3]:
def 스코어뽑기(행번호=None, 폴더='data/model'):
    """저장해 둔 모델 5개로 재방문 확률 p 를 다시 계산함.

    평소에는 안 씀 — p 와 편차가 이미 df 에 들어 있음.
    설명 변수나 모델을 바꿨을 때만 돌림.

    인자
      행번호 : 계산할 df 의 행 번호. 비우면 전체 (1,721만 행이라 20분쯤 걸림)
      폴더   : lgb_fold0~4.txt 와 isotonic.pkl 이 있는 곳

    ⚠️ 이 함수는 평가 기간용임 (모델 5개의 평균).
       학습 기간 행은 '그 유저를 안 본 모델 하나' 로 구해야 해서 Modeling.ipynb 4절을 써야 함
    """
    import pickle

    import lightgbm as lgb

    시작 = time.time()
    쓸행 = df if 행번호 is None else df.iloc[행번호]

    부스터 = []
    for k in range(5):
        부스터.append(lgb.Booster(model_file='{}/lgb_fold{}.txt'.format(폴더, k)))
    with open('{}/isotonic.pkl'.format(폴더), 'rb') as f:
        이소토닉 = pickle.load(f)

    변수이름 = 부스터[0].feature_name()
    X = 쓸행[변수이름]

    p = np.zeros(len(X))
    for k in range(5):
        원점수 = 부스터[k].predict(X)
        p = p + 이소토닉[k].predict(원점수)     # 이소토닉으로 확률 눈금을 맞춤
        print('  조각 {} 끝 ({:.0f}초)'.format(k, time.time() - 시작))
    p = p / 5

    print('{:,}행 — 평균 p {:.4f} (df 에 있던 값 {:.4f})   ({:.0f}초)'.format(
        len(X), p.mean(), 쓸행['p'].mean(), time.time() - 시작))
    return p

## 4. 집단 하나 채점하기

빔서치도 실험도 전부 `점수계산()` 을 부름. 그래서 여기서 보는 숫자와 탐색이 쓰는 숫자가 같음.

```
기울기  = 학습 기간에서 (편차 ~ 날짜) 직선의 기울기
gap     = 평가 평균날짜 − 학습 평균날짜
기준선  = 학습 평균 편차 + 기울기 × gap      ← 하락이 없어도 나오는 값
보정후  = 평가 평균 편차 − 기준선             ← **음수면 하락**
t       = 보정후 ÷ √(평가 편차 오차² + 기준선 오차²)   ← 역시 음수면 하락
```

### 조건 이름 찾기 — `조건표()`

```python
조건표()                # 290개 전부 (축 · 이름 · 값 · 학습행 · 평가행)
조건표('age')           # 이름에 age 가 든 것만
조건표('act_days')      # 축 하나만
```

표에서 **`이름` 칸을 복사해** 넣으면 됩니다. `≤` `≠` 가 타이핑하기 어려우면
**`<=` `!=` 로 써도 알아듣습니다.** 띄어쓰기도 무시합니다.

```python
집단점수('act_days <= 20')     # = 'act_days ≤20'
집단점수('channel != 4')       # = 'channel ≠ 4'
```

이름을 틀리면 비슷한 조건을 알려 줍니다.

### 쓰는 법

```python
집단점수('age = 20s')
집단점수(['is_cancel = 예', 'act_days ≤10'])
집단점수('age = 20s', 보정=False)          # 집단별 기준선 대신 전체 기준선
```

In [4]:
def 점수계산(합, 전체기준선, 보정=True, 최소보정행=10000, 최소인원=100):
    """집단 하나의 합계 12개로 점수를 냄. 빔서치도 실험도 전부 이 함수를 부름.

      기울기  = 학습 기간에서 (편차 ~ 날짜) 직선의 기울기
      gap     = 평가 평균날짜 − 학습 평균날짜
      기준선  = 학습 평균 편차 + 기울기 × gap     ← 하락이 없어도 나오는 값
      보정후  = 평가 평균 편차 − 기준선            ← 음수면 하락
      t       = 보정후 ÷ √(평가 편차 오차² + 기준선 오차²)
    """
    학습행수 = 합['학습행수']
    평가행수 = 합['평가행수']
    평가인원 = 합['평가인원']
    if 학습행수 < 2 or 평가행수 < 1 or 평가인원 < 최소인원:
        return None

    학습평균편차 = 합['학습편차합'] / 학습행수
    평가평균편차 = 합['평가편차합'] / 평가행수
    학습평균날짜 = 합['학습날짜합'] / 학습행수
    평가평균날짜 = 합['평가날짜합'] / 평가행수
    gap = 평가평균날짜 - 학습평균날짜

    날짜별행수 = 합['날짜별행수']
    날짜별편차합 = 합['날짜별편차합']
    있는날 = 날짜별행수 > 0
    날짜수 = int(있는날.sum())
    날짜흩어짐 = 합['학습날짜제곱합'] - 합['학습날짜합'] ** 2 / 학습행수

    기울기 = 0.0
    기준선 = 전체기준선
    기준선분산 = 0.0
    보정방식 = '전체'
    if 보정 and 평가행수 >= 최소보정행 and 날짜흩어짐 > 0 and 날짜수 >= 3:
        기울기 = (합['학습날짜편차합'] - 합['학습날짜합'] * 합['학습편차합'] / 학습행수) / 날짜흩어짐
        절편 = 학습평균편차 - 기울기 * 학습평균날짜
        기준선 = 절편 + 기울기 * 평가평균날짜
        날짜번호 = np.arange(len(날짜별행수))[있는날]
        날짜평균 = 날짜별편차합[있는날] / 날짜별행수[있는날]
        벗어남 = 날짜평균 - (절편 + 기울기 * 날짜번호)
        분산 = (날짜별행수[있는날] * 벗어남 ** 2).sum() / max(날짜수 - 2, 1)
        기준선분산 = 분산 * (1.0 / 학습행수 + gap ** 2 / 날짜흩어짐)
        보정방식 = '집단별'

    평가분산 = 합['평가편차제곱합'] / 평가행수 - 평가평균편차 ** 2
    평가분산 = max(평가분산, 1e-12)
    보정후 = 평가평균편차 - 기준선
    분모 = np.sqrt(평가분산 / 평가인원 + 기준선분산)

    return {'학습행': int(학습행수), '평가행': int(평가행수), '유효인원': 평가인원,
            '학습평균편차': 학습평균편차 * 100, '평가평균편차': 평가평균편차 * 100,
            '학습평균날짜': 학습평균날짜, '평가평균날짜': 평가평균날짜, 'gap': gap,
            '기울기': 기울기 * 100, '기준선': 기준선 * 100,
            '기준선오차': np.sqrt(기준선분산) * 100,
            '평가오차': np.sqrt(평가분산 / 평가인원) * 100,
            '보정후': 보정후 * 100, 't': 보정후 / 분모, '보정': 보정방식, '날짜수': 날짜수}


def 조건표(키워드=''):
    """조건 290개를 표로 보여 줌. 이름을 복사해 집단점수() · 주입() 에 넣으면 됨.

       조건표()              전부
       조건표('age')         이름에 age 가 든 것만
       조건표('act_days')    축 하나만 보기
    """
    평가 = df['구간'].values == '평가'
    학습칸, 평가칸 = {}, {}                       # 축마다 값별 행 수를 한 번만 세어 둠
    for 축 in 코드:
        칸수 = len(이름[축])
        학습칸[축] = np.bincount(코드[축][~평가], minlength=칸수)
        평가칸[축] = np.bincount(코드[축][평가], minlength=칸수)
    줄 = []
    for c in 조건:
        골라낸값 = [이름[c['축']][i] for i in c['값들']]
        줄.append({'축': c['축'], '이름': c['이름'],
                  '값': ', '.join(골라낸값)[:40],
                  '학습행': int(학습칸[c['축']][c['값들']].sum()),
                  '평가행': int(평가칸[c['축']][c['값들']].sum())})
    표 = pd.DataFrame(줄)
    if 키워드:
        표 = 표[표['이름'].str.contains(키워드, case=False)]
    return 표.reset_index(drop=True)


def _기호맞추기(글자):
    """<= 를 ≤ 로, != 를 ≠ 로 바꾸고 띄어쓰기를 지움 — 타이핑하기 쉽게"""
    for 옛, 새 in [('<=', '≤'), ('>=', '≥'), ('!=', '≠'), ('<>', '≠')]:
        글자 = 글자.replace(옛, 새)
    return 글자.replace(' ', '')


def 집단행(조건이름들):
    """조건 이름 하나 또는 목록 → 그 집단에 드는 행 번호.
       `act_days <= 20` 처럼 써도 `act_days ≤20` 으로 알아들음"""
    if isinstance(조건이름들, str):
        조건이름들 = [조건이름들]
    행 = np.arange(len(df))
    for 조건이름 in 조건이름들:
        찾음 = None
        for c in 조건:
            if _기호맞추기(c['이름']) == _기호맞추기(조건이름):
                찾음 = c
        if 찾음 is None:                       # 못 찾으면 비슷한 이름을 보여 줌
            앞말 = 조건이름.split(' ')[0]
            비슷 = [c['이름'] for c in 조건 if 앞말.lower() in c['이름'].lower()][:8]
            raise AssertionError('그런 조건이 없음: {!r}\n비슷한 것: {}'.format(
                조건이름, 비슷 if 비슷 else '없음 — 조건표() 로 찾아볼 것'))
        행 = 행[찾음['표시'][코드[찾음['축']][행]]]
    return 행


def 집단점수(조건이름들, 편차=None, 보정=True, 최소보정행=10000, 최소인원=100, 출력=True):
    """집단 하나를 채점하고 중간값을 전부 돌려줌.

    인자
      조건이름들 : 조건 이름 하나 또는 목록. 행 번호 배열을 직접 넣어도 됨
      편차       : 쓸 편차 배열. 비우면 df['dev'] (주입 실험은 주입된 배열을 넣음)
      보정       : False 면 집단별 기준선 대신 전체 기준선만 씀 (절제 실험)
      최소보정행 : 평가 행이 이보다 적으면 집단별 기준선을 못 믿어 전체 기준선을 씀
    """
    if 편차 is None:
        편차 = df['dev'].values
    학습개수 = (df['구간'].values == '학습').sum()
    전체기준선 = 편차[학습개수:].mean()

    if isinstance(조건이름들, np.ndarray):
        행 = 조건이름들
    else:
        행 = 집단행(조건이름들)
    학습행 = 행[행 < 학습개수]            # df 가 학습 → 평가 순이라 번호로 가를 수 있음
    평가행 = 행[행 >= 학습개수]
    if len(학습행) < 2 or len(평가행) < 1:
        print('행이 너무 적어 잴 수 없음')
        return None

    날짜 = df['day'].values.astype('float64')
    학습날짜 = 날짜[학습행]
    학습편차 = 편차[학습행]
    평가편차 = 편차[평가행]
    날짜칸 = df['day'].values[:학습개수].max() + 1

    합 = {'학습행수': len(학습행),
         '학습날짜합': 학습날짜.sum(),
         '학습편차합': 학습편차.sum(),
         '학습날짜제곱합': (학습날짜 ** 2).sum(),
         '학습날짜편차합': (학습날짜 * 학습편차).sum(),
         '평가행수': len(평가행),
         '평가인원': df['가중치'].values[평가행].sum(),
         '평가편차합': 평가편차.sum(),
         '평가편차제곱합': (평가편차 ** 2).sum(),
         '평가날짜합': 날짜[평가행].sum(),
         '날짜별행수': np.bincount(학습날짜.astype('int32'), minlength=날짜칸).astype('float64'),
         '날짜별편차합': np.bincount(학습날짜.astype('int32'), weights=학습편차, minlength=날짜칸)}

    결과 = 점수계산(합, 전체기준선, 보정, 최소보정행, 최소인원)
    if 결과 is None:
        print('유효 인원이 {}명 미만이라 채점하지 않음'.format(최소인원))
        return None
    결과['행번호'] = 행

    if 출력:
        제목 = 조건이름들 if isinstance(조건이름들, str) else (
            '행 직접 지정' if isinstance(조건이름들, np.ndarray) else ' AND '.join(조건이름들))
        print('집단: {}'.format(제목))
        print('  행        학습 {:>10,} / 평가 {:>10,}      유효 인원 {:,.0f}명'.format(
            결과['학습행'], 결과['평가행'], 결과['유효인원']))
        print('  날짜      학습 평균 {:.2f} / 평가 평균 {:.2f}      gap {:.2f}일'.format(
            결과['학습평균날짜'], 결과['평가평균날짜'], 결과['gap']))
        print('  학습      평균 편차 {:+.3f}%p      기울기 {:+.4f}%p/일'.format(
            결과['학습평균편차'], 결과['기울기']))
        print('  기준선    {:+.3f}%p  ± {:.3f}      ({} 기준선)'.format(
            결과['기준선'], 결과['기준선오차'], 결과['보정']))
        print('  평가      실제 편차 {:+.3f}%p  ± {:.3f}'.format(
            결과['평가평균편차'], 결과['평가오차']))
        print('  보정후 {:+.3f}%p     t = {:+.2f}      (음수면 하락)'.format(
            결과['보정후'], 결과['t']))
    return 결과

## 5. 빔 서치

조건을 하나씩 붙여 가며 t 가 높은 집단만 남김.

```
1단계  조건 225개  → 상위 20 유지
2단계  20 × 조건   → 상위 20 유지
3단계  20 × 조건   → 상위 20 유지  →  겹치는 가설 합쳐 상위 10개
```

| 인자 | 기본값 | 바꾸면 |
|---|---|---|
| `편차` | `df['dev']` | 주입된 배열을 넣으면 그 데이터로 탐색 |
| `빔폭` | 20 | 단계마다 남길 집단 수 |
| `깊이` | 3 | 조건 최대 개수 |
| `최소인원` | 100 | 이보다 작은 집단은 버림 |
| `최소보정행` | 10000 | 평가 행이 이보다 적으면 전체 기준선을 씀 |
| `보정` | True | **False = 절제 실험** |
| `상위` | 10 | 표로 돌려줄 개수 |

돌려주는 것은 `표, 가설목록, 최저t` 세 개.

**하락은 음수**라서 t 가 **가장 작은(가장 음수인)** 집단이 1위임.

주입 없이 돌리면 **최저 t = −7.47**. 실제 데이터에는 하락이 없으니 전부 거짓 경보이고,
실험에서 "이 선보다 더 내려가야 찾았다고 할 수 있다"는 기준으로 씀.

In [5]:
def 빔서치(편차=None, 빔폭=20, 깊이=3, 최소인원=100, 최소보정행=10000,
         무의미컷=0.95, 보정=True, 겹침한도=0.5, 상위=10, 출력=True):
    """조건을 하나씩 붙여 가며 t 가 높은 집단을 찾음. 한 판에 약 40초.

    인자
      편차       : 쓸 편차 배열. 비우면 df['dev'] (주입 실험은 주입된 배열을 넣음)
      빔폭       : 단계마다 남길 상위 집단 수
      깊이       : 조건을 최대 몇 개까지 붙일지. 같은 축은 두 번 안 씀
      최소인원   : 유효 인원이 이보다 적은 집단은 버림
      최소보정행 : 평가 행이 이보다 적으면 집단별 기준선 대신 전체 기준선
      무의미컷   : 조건을 붙여도 집단이 이 비율보다 덜 줄면 버림
      보정       : False 면 전체 기준선만 씀 (절제 실험)
      겹침한도   : 결과를 추릴 때 행이 이만큼 겹치면 같은 가설로 봄
      상위       : 표로 돌려줄 개수

    돌려주는 것 : (표, 가설목록, 최저t).  하락은 음수이므로 t 가 작을수록 센 신호임
    덧붙여 전역 `단계문턱` 에 단계별 빔 20칸 문턱 t 를 남김 (9-9 가 2단계 문턱을 씀)
    """
    시작 = time.time()
    if 편차 is None:
        편차 = df['dev'].values
    학습개수 = (df['구간'].values == '학습').sum()
    전체기준선 = 편차[학습개수:].mean()
    날짜 = df['day'].values.astype('float64')
    가중치 = df['가중치'].values
    날짜칸 = df['day'].values[:학습개수].max() + 1

    def 축별합계(행):
        """집단 안에서 축마다 '값별 합계'를 한 번에 구해 둠.
           이렇게 해 두면 조건 290개를 하나씩 볼 때 1,700만 행을 다시 훑지 않아도 됨"""
        학습행 = 행[행 < 학습개수]
        평가행 = 행[행 >= 학습개수]
        학습날짜 = 날짜[학습행]
        학습편차 = 편차[학습행]
        평가편차 = 편차[평가행]
        날짜번호 = 학습날짜.astype('int32')
        결과 = {}
        for 축 in 코드:
            칸 = 코드[축].max() + 1
            학습코드 = 코드[축][학습행]
            평가코드 = 코드[축][평가행]
            섞은번호 = 학습코드.astype('int32') * 날짜칸 + 날짜번호   # (값, 날짜) 를 한 번호로
            결과[축] = {
                '학습행수': np.bincount(학습코드, minlength=칸).astype('float64'),
                '학습날짜합': np.bincount(학습코드, weights=학습날짜, minlength=칸),
                '학습편차합': np.bincount(학습코드, weights=학습편차, minlength=칸),
                '학습날짜제곱합': np.bincount(학습코드, weights=학습날짜 ** 2, minlength=칸),
                '학습날짜편차합': np.bincount(학습코드, weights=학습날짜 * 학습편차, minlength=칸),
                '평가행수': np.bincount(평가코드, minlength=칸).astype('float64'),
                '평가인원': np.bincount(평가코드, weights=가중치[평가행], minlength=칸),
                '평가편차합': np.bincount(평가코드, weights=평가편차, minlength=칸),
                '평가편차제곱합': np.bincount(평가코드, weights=평가편차 ** 2, minlength=칸),
                '평가날짜합': np.bincount(평가코드, weights=날짜[평가행], minlength=칸),
                '날짜별행수': np.bincount(섞은번호, minlength=칸 * 날짜칸).reshape(칸, 날짜칸) * 1.0,
                '날짜별편차합': np.bincount(섞은번호, weights=학습편차,
                                      minlength=칸 * 날짜칸).reshape(칸, 날짜칸)}
        return 결과

    # 단계마다 '빔 20칸에 들려면 필요한 t' 를 남겨 둠. 조건 2개는 1단계 문턱만 있으면
    # 되는데 조건 3개는 2단계 문턱도 필요해서임 (9-9, 10/12). 전역으로 두는 이유는
    # 돌려주는 값을 바꾸면 실험() · 한판() 이 전부 깨지기 때문임
    global 단계문턱
    단계문턱 = []
    빔 = [([], np.arange(len(df)))]       # (쓴 조건 목록, 그 집단의 행 번호)
    모든가설 = []
    후보수 = 0

    for 단계 in range(깊이):
        후보 = {}
        for 쓴조건, 행 in 빔:
            합계 = 축별합계(행)
            쓴축 = [c['축'] for c in 쓴조건]
            for c in 조건:
                if c['축'] in 쓴축:
                    continue
                고른값 = c['표시']
                합 = {}
                for 키 in 합계[c['축']]:
                    값 = 합계[c['축']][키][고른값]
                    합[키] = 값.sum(axis=0) if 값.ndim == 2 else 값.sum()
                if 합['학습행수'] + 합['평가행수'] > 무의미컷 * len(행):
                    continue                                   # 거의 아무것도 안 거르는 조건
                점수 = 점수계산(합, 전체기준선, 보정, 최소보정행, 최소인원)
                if 점수 is None:
                    continue
                이름들 = tuple(sorted([x['이름'] for x in 쓴조건] + [c['이름']]))
                if 이름들 not in 후보 or 점수['t'] > 후보[이름들][0]['t']:
                    후보[이름들] = (점수, 쓴조건 + [c], 행, c)

        줄세움 = sorted(후보.values(), key=lambda x: x[0]['t'])   # 음수가 센 신호
        후보수 = 후보수 + len(후보)
        for 점수, 조건목록, _, _ in 줄세움:
            모든가설.append((점수, 조건목록))

        단계문턱.append(줄세움[min(빔폭, len(줄세움)) - 1][0]['t'] if 줄세움 else 0.0)
        빔 = []
        for 점수, 조건목록, 부모행, c in 줄세움[:빔폭]:
            빔.append((조건목록, 부모행[c['표시'][코드[c['축']][부모행]]]))

        if 출력:
            print('{}단계 — 후보 {:,}개 / 최저 t {:.2f}   ({:.0f}초)'.format(
                단계 + 1, len(후보), 줄세움[0][0]['t'] if 줄세움 else 0, time.time() - 시작))

    모든가설.sort(key=lambda x: x[0]['t'])
    최저t = 모든가설[0][0]['t']

    # 행이 많이 겹치는 가설은 하나만 남김
    가설목록 = []
    남긴행 = []
    for 점수, 조건목록 in 모든가설[:150]:
        행 = 집단행([c['이름'] for c in 조건목록])
        겹침 = 0.0
        for 옛행 in 남긴행:
            같이 = len(np.intersect1d(행, 옛행, True))
            겹침 = max(겹침, 같이 / (len(행) + len(옛행) - 같이))
        if 겹침 >= 겹침한도:
            continue
        점수 = dict(점수)
        점수['가설'] = ' AND '.join([c['이름'] for c in 조건목록])
        점수['조건수'] = len(조건목록)
        점수['행번호'] = 행
        가설목록.append(점수)
        남긴행.append(행)
        if len(가설목록) >= 상위:
            break

    표 = pd.DataFrame([{'가설': g['가설'], '조건': g['조건수'], '인원': g['유효인원'],
                       '보정후(%p)': g['보정후'], '±기준선': g['기준선오차'],
                       '기울기': g['기울기'], 't': g['t']} for g in 가설목록])
    if 출력:
        print('후보 {:,}개 / 최저 t {:.2f} / 전체 기준선 {:+.3f}%p   ({:.0f}초)'.format(
            후보수, 최저t, 전체기준선 * 100, time.time() - 시작))
        print(표.to_string(formatters={'인원': '{:,.0f}'.format}, float_format='{:+.2f}'.format))
    return 표, 가설목록, 최저t

## 6. 주입 실험

편차 = d1 − p 라서 재방문 1 을 0 으로 뒤집으면 그 행의 편차가 정확히 1.0 내려감.
집단 평가 행의 δ% 를 뒤집으면 집단 평균이 정확히 δ 만큼 내려감.
**학습 기간은 안 건드리므로 모델도 기준선 기울기도 그대로 둬도 됨.**

```python
결과 = 실험('channel = other', [3, 5, 8])
결과 = 실험(['channel = other', 'city = city 15'], [2, 5], 반복=3)
결과 = 실험('channel = other', [5], 보정=False)      # 빔서치 인자를 그대로 넘김
```

| 칸 | 뜻 |
|---|---|
| `최저t` | 그 판 1위 가설의 t (**음수가 셈**). ⚠️ 1위가 정답이 아닐 수 있음 |
| `정답t` | 정답 집단 자신의 t |
| `순위` | 상위 목록에서 정답과 행이 절반 이상 겹친 첫 가설의 등수 (0 = 못 찾음) |
| `최고겹침` | 상위 목록이 정답과 겹친 최대 비율 (1.00 = 정답을 그대로 집어냄) |
| `판정선` | 주입 없이 돌린 최저 t (−7.47). 이 판의 합격 기준 |
| `넘음` | **정답 집단**의 t 가 판정선보다 아래인가 = **그 집단을 보고할 수 있나** |
| `1위넘음` | 그 판 1위(`최저t`)가 판정선보다 아래인가. 1위가 남일 수 있어 참고용 |

In [6]:
def 주입(조건이름들, 델타, seed=20261009):
    """정답 집단의 **평가 기간 행**에서 재방문을 델타(%p) 만큼 1 → 0 으로 뒤집은 편차 배열을 돌려줌.

    편차 = d1 − p 라서 1 을 0 으로 뒤집으면 그 행의 편차가 정확히 1.0 내려감.
    집단 평가 행의 델타% 를 뒤집으면 집단 평균 편차가 정확히 델타만큼 내려감.
    학습 기간은 건드리지 않으므로 기대 모델도 기준선 기울기도 그대로 둬도 됨.
    """
    학습개수 = (df['구간'].values == '학습').sum()
    행 = 집단행(조건이름들) if not isinstance(조건이름들, np.ndarray) else 조건이름들
    평가행 = 행[행 >= 학습개수]
    돌아온행 = 평가행[df['d1'].values[평가행] > 0.5]
    뒤집을수 = int(round(델타 / 100.0 * len(평가행)))
    if 뒤집을수 > len(돌아온행):
        print('돌아온 사람이 모자라 {}%p 는 못 뒤집음'.format(델타))
        return None
    고른행 = np.random.default_rng(seed).choice(돌아온행, 뒤집을수, replace=False)
    편차 = df['dev'].values.copy()
    편차[고른행] = 편차[고른행] - 1.0
    return 편차


def 실험(정답조건, 델타들, 반복=1, seed=20261009, 판정선=None, 저장=None, **빔인자):
    """정답 집단에 하락을 심고 빔 서치가 찾아내는지 표로 돌려줌.

    인자
      정답조건 : 조건 이름 하나, 또는 이름 여러 개의 목록
                 예) 'channel = other'   또는   ['channel = other', 'city = city 15']
      델타들   : 심을 하락 폭(%p) 목록. 예) [3, 5, 8]
      반복     : 같은 조합을 몇 번 돌릴지 (seed 를 1씩 올려 가며). 1회는 우연일 수 있음
      판정선   : 주입 없이 돌린 최저 t. 비우면 여기서 한 판 더 돌려 구함 (40초)
      저장     : csv 경로를 주면 저장
      빔인자   : 빔서치() 인자를 그대로 넘김. 예) 보정=False

    표의 칸
      최저t   그 판 1위 가설의 t (음수가 셈)  /  정답t  정답 집단 자신의 t
      순위    상위 목록에서 정답과 행이 절반 이상 겹친 첫 가설의 등수 (0 = 못 찾음)
      최고겹침  상위 목록이 정답과 겹친 최대 비율 (1.00 = 정답을 그대로 집어냄)
      넘음    정답 집단의 t 가 판정선보다 아래인가 (= 그 집단을 보고할 수 있나)
      1위넘음  그 판 1위(최저t)가 판정선보다 아래인가. 1위가 남일 수 있어 참고용임
    """
    시작 = time.time()
    if isinstance(정답조건, str):
        정답조건 = [정답조건]
    if 판정선 is None:
        _, _, 판정선 = 빔서치(출력=False, **빔인자)
        print('판정선 = {:.2f} (주입 없이 돌린 최저 t)'.format(판정선))

    줄들 = []
    for 조건이름 in 정답조건:
        정답행 = 집단행(조건이름)
        for 델타 in 델타들:
            for 회 in range(반복):
                편차 = 주입(조건이름, 델타, seed + 회)
                if 편차 is None:
                    continue
                표, 가설목록, 최저t = 빔서치(편차=편차, 출력=False, **빔인자)
                정답 = 집단점수(정답행, 편차=편차, 출력=False,
                             보정=빔인자.get('보정', True),
                             최소보정행=빔인자.get('최소보정행', 10000))
                순위 = 0
                최고겹침 = 0.0
                for i in range(len(가설목록)):
                    행 = 가설목록[i]['행번호']
                    같이 = len(np.intersect1d(행, 정답행, True))
                    겹침 = 같이 / (len(행) + len(정답행) - 같이)
                    최고겹침 = max(최고겹침, 겹침)
                    if 겹침 >= 0.5 and 순위 == 0:
                        순위 = i + 1
                줄들.append({'집단': 조건이름, 'δ': 델타, '회': 회 + 1,
                           '최저t': 최저t, '정답t': 정답['t'], '정답보정후': 정답['보정후'],
                           '순위': 순위, '최고겹침': 최고겹침, '판정선': 판정선,
                           '넘음': 정답['t'] < 판정선,           # 우리 집단을 보고할 수 있나
                           '1위넘음': 최저t < 판정선})          # 그 판 1위 기준 (남일 수 있음)
                print('  {:<28s} δ={:>3}%p {}회   최저t {:>6.2f}  정답t {:>6.2f}  '
                      '순위 {:<3s} 겹침 {:.2f}   ({:.0f}초)'.format(
                          조건이름, 델타, 회 + 1, 최저t, 정답['t'],
                          str(순위) if 순위 else '-', 최고겹침, time.time() - 시작))

    결과표 = pd.DataFrame(줄들)
    if 저장:
        결과표.to_csv(저장, index=False, encoding='utf-8-sig')
        print('저장: ' + 저장)
    return 결과표

## 8. 탐지 하한 곡선 (구현 5번 본실험)

**δ 를 얼마나 키워야 찾아내는지**를 집단 크기별로 잰다. 세 집단은 조건 하나로 표현되고,
무주입 `|t| < 1` 로 조용하며, **보정 문턱(2일 창에서 평가 1,400행)을 걸치도록** 크기를 벌려 골랐다.

| | 값 | 왜 |
|---|---|---|
| 정답 집단 | 고정 3개 | 무주입에서 조용한 집단만 — 무작위로 뽑으면 이미 떠 있는 집단이 섞임 |
| δ 격자 | 집단마다 **예상 하한 근처 5단계** | 9-3 의 보고 하한(상관 0.998)으로 미리 가늠함 |
| 반복 | **2회** | 10회를 돌려 보니 순위가 글자 그대로 같았음(10/10). 경계 근처만 더 주면 됨 |
| 판 수 | 3 × 5 × 2 = **30판** | 한 판 약 50초 → **약 25분** |

집단별로 끊어 **`results/`** 에 창 이름을 붙여 저장한다(`하한_소_2일.csv` 꼴).

### 결과 읽기

**`넘음` = 정답 집단의 t 가 판정선보다 아래인가** = 그 집단을 보고할 수 있는가.
`최저t` 는 그 판 **1위**의 t 인데, δ 가 작을 때는 무주입 때의 거짓 경보가 1위를 지키므로
**우리 집단 숫자가 아닐 수 있다.**

In [7]:
# ── 8-0. 판정선 구하기 (1분) ─────────────────────────────────────────────
import os

os.makedirs('results', exist_ok=True)

_, _, 판정선 = 빔서치(최소보정행=보정행, 출력=False)
print('판정선 {:.3f}   (주입 없이 돌린 최저 t)'.format(판정선))

판정선 -3.636   (주입 없이 돌린 최저 t)


In [ ]:
# ── 8-1. 소 집단 — 평가 453행 (보정 문턱 1,400 아래라 전체 기준선을 씀) ──
# ⚠️ 2일 창에서는 payment_method_id = 18 을 못 씀 — 무주입 t 가 +0.11 → +1.71 로
#    떠올라 「조용한 집단」 기준(|t|<1)을 벗어남. 출발점이 +1.71 이면 그만큼 더 심어야 해
#    하한이 과대평가됨. 같은 자리(게이트 아래)의 조용한 집단으로 바꿈 (t −0.05)
# 10판 ≈ 8분
소 = 실험('payment_method_id = 26', [5, 7, 9, 11, 14], 반복=2,
        판정선=판정선, 저장=저장경로('하한_소'), 최소보정행=보정행)

  payment_method_id = 26       δ=  5%p 1회   최저t  -3.64  정답t  -2.01  순위 -   겹침 0.43   (49초)
  payment_method_id = 26       δ=  5%p 2회   최저t  -3.64  정답t  -2.05  순위 -   겹침 0.43   (98초)
  payment_method_id = 26       δ=  7%p 1회   최저t  -3.72  정답t  -2.70  순위 1   겹침 0.70   (142초)
  payment_method_id = 26       δ=  7%p 2회   최저t  -3.64  정답t  -2.72  순위 3   겹침 0.61   (187초)
  payment_method_id = 26       δ=  9%p 1회   최저t  -4.61  정답t  -3.39  순위 1   겹침 0.73   (229초)
  payment_method_id = 26       δ=  9%p 2회   최저t  -4.29  정답t  -3.42  순위 1   겹침 0.72   (273초)
  payment_method_id = 26       δ= 11%p 1회   최저t  -4.58  정답t  -4.02  순위 1   겹침 0.81   (316초)
  payment_method_id = 26       δ= 11%p 2회   최저t  -4.75  정답t  -4.07  순위 1   겹침 0.65   (356초)
  payment_method_id = 26       δ= 14%p 1회   최저t  -5.70  정답t  -4.83  순위 1   겹침 0.73   (396초)
  payment_method_id = 26       δ= 14%p 2회   최저t  -5.68  정답t  -4.92  순위 1   겹침 0.77   (436초)
저장: results/하한_소_2일.csv


In [45]:
# ── 8-2. 중 집단 — 평가 3,989행 (무주입 t +0.67, 집단별 보정) ───────────
# 10판 ≈ 8분
중 = 실험('channel = other', [2, 3, 4, 5, 7], 반복=2,
        판정선=판정선, 저장=저장경로('하한_중'), 최소보정행=보정행)

  channel = other              δ=  2%p 1회   최저t  -3.63  정답t  -1.49  순위 -   겹침 0.00   (47초)
  channel = other              δ=  2%p 2회   최저t  -3.63  정답t  -1.49  순위 -   겹침 0.00   (100초)
  channel = other              δ=  3%p 1회   최저t  -3.63  정답t  -2.54  순위 3   겹침 0.66   (146초)
  channel = other              δ=  3%p 2회   최저t  -3.72  정답t  -2.53  순위 -   겹침 0.33   (188초)
  channel = other              δ=  4%p 1회   최저t  -4.40  정답t  -3.54  순위 3   겹침 0.67   (236초)
  channel = other              δ=  4%p 2회   최저t  -4.68  정답t  -3.54  순위 4   겹침 0.66   (281초)
  channel = other              δ=  5%p 1회   최저t  -4.95  정답t  -4.51  순위 1   겹침 0.67   (326초)
  channel = other              δ=  5%p 2회   최저t  -5.17  정답t  -4.50  순위 2   겹침 0.67   (374초)
  channel = other              δ=  7%p 1회   최저t  -6.55  정답t  -6.42  순위 1   겹침 0.76   (419초)
  channel = other              δ=  7%p 2회   최저t  -6.53  정답t  -6.44  순위 1   겹침 0.99   (465초)
저장: results/하한_중_2일.csv


In [46]:
# ── 8-3. 대 집단 — 평가 74,560행 (무주입 t +0.27, 집단별 보정) ──────────
# 10판 ≈ 9분
대 = 실험('city = city 15', [0.6, 0.9, 1.2, 1.6, 2.2], 반복=2,
        판정선=판정선, 저장=저장경로('하한_대'), 최소보정행=보정행)

  city = city 15               δ=0.6%p 1회   최저t  -3.75  정답t  -1.73  순위 6   겹침 0.53   (46초)
  city = city 15               δ=0.6%p 2회   최저t  -3.68  정답t  -1.73  순위 7   겹침 0.53   (90초)
  city = city 15               δ=0.9%p 1회   최저t  -4.95  정답t  -2.72  순위 -   겹침 0.48   (134초)
  city = city 15               δ=0.9%p 2회   최저t  -4.81  정답t  -2.73  순위 10  겹침 0.73   (190초)
  city = city 15               δ=1.2%p 1회   최저t  -6.17  정답t  -3.72  순위 2   겹침 0.57   (246초)
  city = city 15               δ=1.2%p 2회   최저t  -6.24  정답t  -3.72  순위 2   겹침 0.57   (303초)
  city = city 15               δ=1.6%p 1회   최저t  -7.73  정답t  -5.03  순위 1   겹침 0.64   (367초)
  city = city 15               δ=1.6%p 2회   최저t  -7.59  정답t  -5.03  순위 2   겹침 0.63   (442초)
  city = city 15               δ=2.2%p 1회   최저t  -9.89  정답t  -6.98  순위 1   겹침 0.64   (524초)
  city = city 15               δ=2.2%p 2회   최저t  -9.75  정답t  -6.98  순위 1   겹침 0.77   (607초)
저장: results/하한_대_2일.csv


In [47]:
# ── 8-4. 합쳐서 요약 ────────────────────────────────────────────────────
곡선 = pd.concat([소, 중, 대], ignore_index=True)
곡선['Hit@1'] = (곡선['순위'] == 1).astype(int)
곡선['Hit@3'] = ((곡선['순위'] >= 1) & (곡선['순위'] <= 3)).astype(int)
곡선.to_csv(저장경로('하한_전체'), index=False, encoding='utf-8-sig')

요약 = 곡선.groupby(['집단', 'δ']).agg(
    판수=('회', 'count'), Hit1=('Hit@1', 'mean'), Hit3=('Hit@3', 'mean'),
    최고겹침=('최고겹침', 'mean'), 정답t=('정답t', 'mean'),
    정답t표준편차=('정답t', 'std')).round(3)
print(요약.to_string())

print('\n판정선 {:.2f} — 정답t 가 이보다 아래인 δ 가 보고 가능한 하한'.format(판정선))
for 이름, g in 곡선.groupby('집단', sort=False):
    찾음 = g[g['Hit@3'] == 1]['δ']
    보고 = g[(g['Hit@3'] == 1) & (g['정답t'] < 판정선)]['δ']
    print('  {:<26s} 찾음δ {} / 보고δ {}'.format(
        이름, 찾음.min() if len(찾음) else '-', 보고.min() if len(보고) else '-'))

                             판수  Hit1  Hit3   최고겹침    정답t  정답t표준편차
집단                     δ                                          
channel = other        2.0    2   0.0   0.0  0.003 -1.493    0.000
                       3.0    2   0.0   0.5  0.493 -2.534    0.004
                       4.0    2   0.0   0.5  0.665 -3.539    0.002
                       5.0    2   0.5   1.0  0.675 -4.504    0.009
                       7.0    2   1.0   1.0  0.872 -6.430    0.017
city = city 15         0.6    2   0.0   0.0  0.531 -1.728    0.000
                       0.9    2   0.0   0.0  0.603 -2.725    0.000
                       1.2    2   0.0   1.0  0.568 -3.716    0.000
                       1.6    2   0.5   1.0  0.638 -5.028    0.001
                       2.2    2   1.0   1.0  0.706 -6.983    0.002
payment_method_id = 26 5.0    2   0.0   0.0  0.428 -2.026    0.027
                       7.0    2   0.5   1.0  0.656 -2.711    0.015
                       9.0    2   1.0   1.0  0.723 -3.404    0

## 9. 집단을 늘려 — 하한을 크기의 함수로

8절은 **손으로 고른 집단 3개**로 하한을 쟀다. 그런데 셋은 크기만 다른 게 아니라 축도 다르고
기준선 보정 방식도 다르다(소는 평가 4,381행이라 1만 미만 → 전체 기준선, 중·대는 집단별).
그래서 **"하한이 크기 때문에 갈렸다"고 말할 근거가 약하다.**

여기서는 집단을 많이 써서 그 관계를 선으로 본다. 핵심은 **싸게 재는 방법**이다.

| 재는 것 | 필요한 것 | 한 집단당 |
|---|---|---|
| **보고 하한** (정답t 가 판정선 아래로 내려가는 δ) | **그 집단 자신의 t 만** | **약 0.5초** |
| 찾음 하한 (빔 서치 상위에 올라오는 δ) | 빔 서치 한 판 | 약 70초 |

보고 하한은 **빔 서치가 필요 없다.** 그래서 조건 290개를 **전부** 재도 몇 분이면 끝난다.
게다가 정답t 가 δ 에 거의 직선이라(10/10 확인, %p당 0.44 · 표준편차 0.03) **δ 두 점만 재서
직선으로 이으면** 하한이 나온다.

**먼저 8-0 을 돌려 `판정선` 이 있어야 한다.**

In [8]:
# ── 9-1. 조건 290개를 주입 없이 전부 채점 (약 2분) ──────────────────────
# 두 가지를 한꺼번에 봄
#   ① 무주입 t 가 집단마다 어떻게 퍼져 있나 — 0 근처가 흔한지, 꼬리가 얼마나 두꺼운지
#   ② 정답 집단으로 쓸 '조용한' 집단 고르기 — 출발점이 0 이라야 시험이 공정함
#      (출발점이 −0.8 이면 공짜로 0.8 을 먹고 들어가 하한이 좋아 보임)
시작 = time.time()
줄 = []
for c in 조건:
    r = 집단점수(c['이름'], 최소보정행=보정행, 출력=False)
    if r is None:                      # 유효 인원 100명 미만은 빠짐
        continue
    분모 = (r['평가오차'] ** 2 + r['기준선오차'] ** 2) ** 0.5
    줄.append({'이름': c['이름'], '축': c['축'],
               '평가행': r['평가행'], '유효인원': r['유효인원'],
               't': r['t'], '보정후': r['보정후'], '보정': r['보정'],
               '기울기': r['기울기'], '분모': 분모,
               # t 가 판정선에 닿으려면 δ 가 얼마여야 하는지 — 종이 계산
               '예상하한': r['보정후'] + abs(판정선) * 분모})

무주입 = pd.DataFrame(줄).sort_values('t').reset_index(drop=True)
print('채점 {}개 / 인원 미달로 빠짐 {}개   ({:.0f}초)'.format(
    len(무주입), len(조건) - len(무주입), time.time() - 시작))

print('\n[무주입 t 가 어떻게 퍼져 있나]')
print(무주입['t'].describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_string())

칸 = pd.cut(무주입['t'].abs(), [0, 1, 2, 3, 5, 100],
            labels=['|t|<1 (조용)', '1~2', '2~3', '3~5', '5 이상'])
print('\n' + 칸.value_counts().sort_index().to_string())

print('\n[가장 센 10개] — 값이 고르게 늘어서 있으면(고원) 판정선이 안정적이고,')
print('                1등만 뚝 떨어져 있으면 한 번 뽑은 판정선에 기대는 것이 위태로움')
print(무주입.head(10)[['이름', '축', '평가행', 't', '보정후', '보정']].round(2).to_string(index=False))

유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
채점 264개 / 인원 미달로 빠짐 4개   (117초)

[무주입 t 가 어떻게 퍼져 있나]
count    264.00
mean       0.82
std        0.86
min       -1.43
5%        -0.82
25%        0.22
50%        1.07
75%        1.32
95%        2.13
max        3.14

t
|t|<1 (조용)    115
1~2           125
2~3            23
3~5             1
5 이상            0

[가장 센 10개] — 값이 고르게 늘어서 있으면(고원) 판정선이 안정적이고,
                1등만 뚝 떨어져 있으면 한 번 뽑은 판정선에 기대는 것이 위태로움
                    이름                  축    평가행     t   보정후  보정
payment_method_id = 19  payment_method_id    997 -1.43 -2.31  전체
             age = 10s                age  62058 -1.33 -0.40 집단별
   plan_list_price ≤99    plan_list_price 265717 -1.28 -0.20 집단별
payment_method_id = 13  payment_method_id   7338 -1.26 -0.75 집단별
actual_amount_paid ≤99 actual_amount_paid 267947 -1.21 -0.19 집단별
              trend >3              trend 151383 -1.10 -0.25 집단별
payment_method_id = 15  payment_method_i

In [9]:
# ── 9-2. 정답 집단 고르기 — 조용하고(|t|<1) 크기가 고르게 퍼지도록 ──────
# 크기 경계에 1만을 넣었음. 평가 행 1만이 '집단별 기준선 보정'을 받는 문턱이라
# 그 양쪽을 갈라 봐야 '크기 때문인지 보정 방식 때문인지' 를 구분할 수 있음
조용 = 무주입[무주입['t'].abs() < 1].copy()
경계 = [0, 3000, 10000, 30000, 100000, 10 ** 9]
이름표 = ['~3천', '3천~1만', '1만~3만', '3만~10만', '10만+']
조용['크기'] = pd.cut(조용['평가행'], 경계, labels=이름표)
print('조용한 집단 {}개 / 전체 {}개'.format(len(조용), len(무주입)))
print(조용['크기'].value_counts().sort_index().to_string())

뽑은줄 = []
for 칸 in 이름표:
    후보 = 조용[조용['크기'] == 칸].copy()
    후보['조용함'] = 후보['t'].abs()
    후보 = 후보.sort_values('조용함')          # 가장 조용한 것부터
    쓴축 = []
    담음 = 0
    for i in range(len(후보)):
        r = 후보.iloc[i]
        if r['축'] in 쓴축:                    # 한 칸에서 같은 축은 하나만
            continue                           # (10/6·10/10 에 결제수단이 표를 뒤덮는 문제가 있었음)
        쓴축.append(r['축'])
        뽑은줄.append(r)
        담음 = 담음 + 1
        if 담음 == 3:
            break

정답집단 = pd.DataFrame(뽑은줄).reset_index(drop=True)
print('\n[고른 정답 집단 {}개]'.format(len(정답집단)))
print(정답집단[['이름', '축', '크기', '평가행', '유효인원', 't', '보정', '예상하한']].round(2).to_string())

조용한 집단 115개 / 전체 264개
크기
~3천        9
3천~1만      5
1만~3만     15
3만~10만    15
10만+      71

[고른 정답 집단 13개]
                             이름                  축      크기     평가행      유효인원     t   보정  예상하한
0        payment_method_id = 26  payment_method_id     ~3천     453     268.5 -0.05   전체  8.46
1        payment_method_id = 28  payment_method_id   3천~1만    4621    2678.0 -0.04  집단별  2.54
2                 is_cancel = 예          is_cancel   3천~1만    9743    6052.0 -0.25  집단별  2.23
3               channel = other            channel   3천~1만    3989    2405.0  0.67  집단별  3.91
4        payment_method_id = 31  payment_method_id   1만~3만   11487    6853.5 -0.17  집단별  1.86
5       payment_plan_days 30~90  payment_plan_days   1만~3만   21033   12281.0  0.45  집단별  1.49
6             days_to_expire 결측     days_to_expire   1만~3만   18915   14270.0 -0.81  집단별  1.32
7              weekend_ratio 결측      weekend_ratio  3만~10만   48757   33493.0  0.00  집단별  1.99
8                   act_days ≤0           act_da

In [10]:
# ── 9-3. 보고 하한을 싸게 재기 (빔 서치 없이) ──────────────────────────
# 보고 하한 = 정답t 가 판정선보다 아래로 내려가는 가장 작은 δ.
# '그 집단 자신의 t' 만 보면 되므로 빔 서치가 필요 없음 (빔 서치는 순위·겹침을 볼 때만 씀).
# 정답t 가 δ 에 거의 직선이라 δ 두 점만 재서 이으면 됨.
def 보고하한(이름, 작은=2.0, 큰=10.0, seed=20261009):
    """정답t 가 판정선에 닿는 δ 를 직선 보간으로 구함. 못 재면 None"""
    행 = 집단행(이름)
    t = []
    for d in (작은, 큰):
        편차 = 주입(행, d, seed)
        if 편차 is None:
            return None
        t.append(집단점수(행, 편차=편차, 최소보정행=보정행, 출력=False)['t'])
    기울기 = (t[1] - t[0]) / (큰 - 작은)        # δ 1%p 당 t 가 내려가는 양 (음수)
    if 기울기 >= 0:
        return None
    return 작은 + (판정선 - t[0]) / 기울기


시작 = time.time()
하한 = []
for i in range(len(무주입)):
    이름 = 무주입['이름'].iloc[i]
    하한.append(보고하한(이름))
    if (i + 1) % 50 == 0:
        print('  {}/{} ({:.0f}초)'.format(i + 1, len(무주입), time.time() - 시작))
무주입['보고하한'] = 하한
print('끝 ({:.0f}초)'.format(time.time() - 시작))

# 종이 계산(예상하한)과 실제로 잰 값이 맞는지 — 맞으면 하한을 크기만으로 예측할 수 있음
맞춤 = 무주입.dropna(subset=['보고하한'])
print('\n예상하한 vs 보고하한 상관 {:.4f}   (1에 가까울수록 종이 계산이 맞는 것임)'.format(
    맞춤['예상하한'].corr(맞춤['보고하한'])))

print('\n[8절에서 실제로 돌린 집단과 맞춰 보기]')
for 이름, 잰값 in [('payment_method_id = 18', '15.7 (8-1 에서 50판으로 잼)'),
                 ('channel = other', '파일럿 5 근처'),
                 ('city = city 15', '파일럿 2 근처')]:
    g = 무주입[무주입['이름'] == 이름]
    if len(g):
        print('  {:<26s} 보고하한 {:>6.2f}%p   ← {}'.format(
            이름, g['보고하한'].iloc[0], 잰값))

# 저장 — 9.4분짜리라 커널이 죽어도 다시 안 돌리게. 논문 그림도 이 파일로 그림
import os
os.makedirs('results', exist_ok=True)
무주입.to_csv(저장경로('무주입_조건'), index=False, encoding='utf-8-sig')
print('\n저장: ' + 저장경로('무주입_조건'))

  50/264 (48초)
  100/264 (103초)
  150/264 (164초)
  200/264 (232초)
  250/264 (291초)
끝 (306초)

예상하한 vs 보고하한 상관 0.9979   (1에 가까울수록 종이 계산이 맞는 것임)

[8절에서 실제로 돌린 집단과 맞춰 보기]
  payment_method_id = 18     보고하한  11.24%p   ← 15.7 (8-1 에서 50판으로 잼)
  channel = other            보고하한   4.23%p   ← 파일럿 5 근처
  city = city 15             보고하한   1.13%p   ← 파일럿 2 근처

저장: results/무주입_조건_2일.csv


In [11]:
# ── 9-4. 크기와 하한의 관계 ─────────────────────────────────────────────
볼것 = 무주입.dropna(subset=['보고하한']).copy()
볼것['크기'] = pd.cut(볼것['평가행'], [0, 3000, 10000, 30000, 100000, 10 ** 9],
                    labels=['~3천', '3천~1만', '1만~3만', '3만~10만', '10만+'])
볼것['조용'] = 볼것['t'].abs() < 1

요약 = 볼것.groupby(['크기', '보정'], observed=True).agg(
    집단수=('이름', 'count'),
    평가행중앙=('평가행', 'median'),
    보고하한중앙=('보고하한', 'median'),
    보고하한최소=('보고하한', 'min'),
    보고하한최대=('보고하한', 'max')).round(2)
print('[크기 구간 × 기준선 보정 방식 별 보고 하한]')
print(요약.to_string())

print('\n[조용한 집단만 — 공정한 시험대]')
print(볼것[볼것['조용']].groupby('크기', observed=True).agg(
    집단수=('이름', 'count'), 보고하한중앙=('보고하한', 'median')).round(2).to_string())

# 유효 인원이 4배면 하한이 반으로 줄어야 함(1/√n). 로그로 재면 기울기가 −0.5 여야 맞음
_x = np.log(볼것['유효인원'].to_numpy())
_y = np.log(볼것['보고하한'].to_numpy())
_기울기 = np.polyfit(_x, _y, 1)[0]
print('\nlog(유효인원) 대 log(보고하한) 기울기 {:.3f}'.format(_기울기))
print('  −0.5 에 가까우면 하한이 1/√(유효 인원) 으로 움직인다는 뜻 — 크기만으로 예측 가능')

[크기 구간 × 기준선 보정 방식 별 보고 하한]
            집단수     평가행중앙  보고하한중앙  보고하한최소  보고하한최대
크기     보정                                        
~3천    전체     9     453.0   10.77    4.05   16.31
       집단별    4    2354.5    4.31    3.96    5.69
3천~1만  집단별    7    7338.0    2.64    1.43    4.23
1만~3만  집단별   21   18911.0    1.32    1.05    4.86
3만~10만 집단별   21   48757.0    2.00    0.46    3.01
10만+   집단별  202  690355.5    0.56   -0.12    1.99

[조용한 집단만 — 공정한 시험대]
        집단수  보고하한중앙
크기                 
~3천       9    8.87
3천~1만     5    2.82
1만~3만    15    1.32
3만~10만   15    2.00
10만+     71    0.49

log(유효인원) 대 log(보고하한) 기울기 nan
  −0.5 에 가까우면 하한이 1/√(유효 인원) 으로 움직인다는 뜻 — 크기만으로 예측 가능


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_480384\2478525810.py:22: RuntimeWarning: invalid value encountered in log
  _y = np.log(볼것['보고하한'].to_numpy())


### 조건 2개짜리 정답 집단 (9-5 ~ 9-6)

지금까지 정답 집단은 **전부 조건 1개**였다. 조건 1개는 빔 서치 1단계에서 290개가 **전부 채점**되므로
도달이 보장된다. 그런데 조건 2개짜리는 **부모(조건 1개)가 1단계에서 빔 20칸에 들어야**
2단계에서 만들어진다. 못 들면 δ 를 아무리 키워도 **영원히 못 찾는다.**

| 말 | 뜻 |
|---|---|
| **부모** | 조건 1개짜리 집단 (예: `channel = other`) |
| **자식** | 거기에 조건을 하나 더 붙인 집단 (부모 안에 들어 있는 더 작은 덩어리) |
| **희석 r** | `자식 평가행 ÷ 더 작은 부모의 평가행` |

하락을 **자식에만** 심으면 부모 평균 편차는 `δ × r` 만큼만 내려간다 (이건 등식이다 —
뒤집은 행 수 ÷ 부모 행 수가 곧 `δ × r` 이다). 즉 **r 이 작으면 부모가 눈치채지 못해
자식까지 못 간다.** 부모가 둘이므로 **둘 중 먼저 빔에 드는 쪽**이 기준이다.

도달δ 는 **부모를 직접 채점해서** 구한다 (10/12 정확식으로 바꿈).

```
도달δ = ( 부모 보정후 + |빔문턱| × 부모 분모 ) ÷ r        ← 두 부모 중 작은 값
```

부모(조건 1개)의 보정후·분모는 **9-1 이 이미 재 뒀으므로** 꺼내 쓰면 된다.

> ⚠️ 전에는 `부모 t ≈ 자식 t × √r` 로 근사했다. 이 근사는 **"분모가 1/√N 으로 준다"** 를
> 깔고 있는데, 이 데이터의 기울기는 **2일 −0.373 / 14일 −0.310** 이다 (기준선 오차에 N 과
> 무관한 바닥이 있어서 — 보고 하한이 `N^(−1/3)` 인 것과 같은 원인). 그래서 근사는 부모를
> 실제보다 민감하게 봐 **도달δ 를 작게 잡았다.** 자세한 것은 『두관문_설명』 5.3~5.4.

고르는 기준 넷 — ① 자식과 **두 부모 모두** 무주입 `|t| < 1` ② 축이 다를 것
③ 유효 인원 100명 이상 ④ **크기와 r 을 격자로 벌릴 것**.

크기와 r 은 따로 조절된다. `자식 크기 ≈ p_A × p_B` (곱) 이고 `r ≈ max(p_A, p_B)` (큰 쪽) 이므로,
같은 크기라도 두 조건의 선택도 배분을 바꾸면 r 이 달라진다.

In [12]:
# ── 9-5. 축이 다른 조건 쌍을 무작위로 채점 (약 10분) ────────────────────
# 9-1 에서 나온 조용한 단일 조건만 재료로 씀 — 부모가 원래 떠 있으면 시험이 불공정함
뽑을쌍 = 1000
난수 = np.random.default_rng(20261011)

# ⚠️ 커널에 남아 있는 무주입 은 **다른 창의 것일 수 있음** (9-1 을 이 창에서 다시 안 돌렸을 때).
#    「조용함」 판정이 창마다 달라지므로 — 2일 창에서 payment_method_id = 18 은 t +0.11 → +1.71 로
#    조용하지 않게 됨 — 반드시 이 창의 파일에서 다시 읽음
무주입 = pd.read_csv(저장경로('무주입_조건'), encoding='utf-8-sig')
# 옛 14일 파일에는 같은 이름이 22종 있음 (10/11 조건 중복을 고치기 전 것).
# 9-3 을 거쳐 새로 쓴 파일에는 없지만, 옛 파일로 이어가도 공정하게 뽑히도록 막아 둠
무주입 = 무주입.drop_duplicates('이름').reset_index(drop=True)
print('{} 에서 조건 {}개를 읽음 (t {:+.2f} ~ {:+.2f})'.format(
    저장경로('무주입_조건'), len(무주입), 무주입['t'].min(), 무주입['t'].max()))

조용 = 무주입[무주입['t'].abs() < 1].reset_index(drop=True)
빔문턱 = 무주입['t'].nsmallest(20).iloc[-1]          # 1단계에서 빔 20칸에 들려면 필요한 t
print('조용한 단일 조건 {}개 / 빔 20칸 문턱 t = {:.2f}'.format(len(조용), 빔문턱))
# 판정선도 창마다 다름 (14일 −7.470 / 2일 −3.636). 아래 예측이 전부 이 값에 달려 있으니 눈으로 확인할 것
print('지금 쓰는 판정선 = {:.3f}  (창 {})'.format(판정선, 창이름))

모든쌍 = []
for i in range(len(조용)):
    for j in range(i + 1, len(조용)):
        if 조용['축'].iloc[i] != 조용['축'].iloc[j]:   # 같은 축은 빔 서치가 안 붙임
            모든쌍.append((i, j))
뽑을쌍 = min(뽑을쌍, len(모든쌍))
고른쌍 = 난수.choice(len(모든쌍), 뽑을쌍, replace=False)
print('축이 다른 쌍 {:,}개 중 {}개를 채점함'.format(len(모든쌍), 뽑을쌍))

시작 = time.time()
줄 = []
for n in range(len(고른쌍)):
    i, j = 모든쌍[고른쌍[n]]
    이름A, 이름B = 조용['이름'].iloc[i], 조용['이름'].iloc[j]
    행 = 집단행([이름A, 이름B])
    점 = 집단점수(행, 최소보정행=보정행, 출력=False)
    if 점 is None:                                    # 유효 인원 100명 미만
        continue
    작은부모 = min(조용['평가행'].iloc[i], 조용['평가행'].iloc[j])
    분모 = (점['평가오차'] ** 2 + 점['기준선오차'] ** 2) ** 0.5
    희석 = 점['평가행'] / 작은부모
    # 도달δ — **부모를 직접 채점한 값**으로 구함 (10/12 정확식으로 바꿈).
    #   전에는 '부모 분모 = 내 분모 × √r' 로 근사했는데, 이 데이터에서 분모는 1/√N 으로
    #   줄지 않아(기울기 2일 −0.373 / 14일 −0.310) 부모를 실제보다 민감하게 봤음.
    #   쓰는 사실은 하나뿐이고 그건 등식임 — 자식 평가행의 δ% 를 뒤집으면
    #   부모 평균 편차가 정확히 δ × r 내려감.
    길 = []
    for 부 in [조용.iloc[i], 조용.iloc[j]]:
        r = 점['평가행'] / 부['평가행']
        길.append((부['보정후'] + abs(빔문턱) * 부['분모']) / r)
    도달 = min(길)                       # 둘 중 하나만 빔에 들면 되므로 싼 쪽
    줄.append({'A': 이름A, 'B': 이름B, '축A': 조용['축'].iloc[i], '축B': 조용['축'].iloc[j],
               '평가행': 점['평가행'], '유효인원': 점['유효인원'], 't': 점['t'],
               '보정후': 점['보정후'], '보정': 점['보정'], '분모': 분모, '희석r': 희석,
               # 예측 둘 — 보고δ 는 자신의 t 가 판정선에 닿는 δ,
               #           도달δ 는 부모가 빔 20칸에 드는 δ. 둘 중 큰 쪽이 실제 하한이 됨
               '보고δ예측': 점['보정후'] + abs(판정선) * 분모,
               '도달δ예측': max(도달, 0.0),      # 음수 = 부모가 심기 전에 이미 빔 안
               '부모이미빔안': 도달 <= 0})
    if (n + 1) % 200 == 0:
        print('  {}/{} ({:.0f}초)'.format(n + 1, 뽑을쌍, time.time() - 시작))

쌍 = pd.DataFrame(줄)
조용쌍 = 쌍[쌍['t'].abs() < 1].reset_index(drop=True)
print('\\n채점 {}개 (인원 미달 {}개 빠짐) / 자식도 조용한 것 {}개   ({:.0f}초)'.format(
    len(쌍), 뽑을쌍 - len(쌍), len(조용쌍), time.time() - 시작))

print('\\n부모가 심기 전에 이미 빔 20칸 안인 쌍 {}개 ({:.1f}%) — 이 쌍은 도달 관문이 없음'.format(
    조용쌍['부모이미빔안'].sum(), 100 * 조용쌍['부모이미빔안'].mean()))
print('도달이 보고보다 큰(= 빔에 막히는) 쌍 {}개 ({:.1f}%)'.format(
    (조용쌍['도달δ예측'] > 조용쌍['보고δ예측']).sum(),
    100 * (조용쌍['도달δ예측'] > 조용쌍['보고δ예측']).mean()))

print('[희석 r 이 어떻게 퍼져 있나]')
print(조용쌍['희석r'].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
print('\\n[자식 평가행]')
print(조용쌍['평가행'].describe(percentiles=[.25, .5, .75]).round(0).to_string())

# 저장 — 12분짜리라 커널이 죽어도 다시 안 돌리게
import os
os.makedirs('results', exist_ok=True)
조용쌍.to_csv(저장경로('쌍_조용'), index=False, encoding='utf-8-sig')
print('저장: ' + 저장경로('쌍_조용'))

results/무주입_조건_2일.csv 에서 조건 264개를 읽음 (t -1.43 ~ +3.14)
조용한 단일 조건 115개 / 빔 20칸 문턱 t = -0.82
지금 쓰는 판정선 = -3.636  (창 2일)
축이 다른 쌍 6,145개 중 1000개를 채점함
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
행이 너무 적어 잴 수 없음
  200/1000 (63초)
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수 없음
유효 인원이 100명 미만이라 채점하지 않음
유효 인원이 100명 미만이라 채점하지 않음
행이 너무 적어 잴 수

In [13]:
# ── 9-6. 희석 정도로 셋으로 나눠 5개씩 (크기는 고르게 퍼지도록) ─────────
# 크기로 거르지 않음. 큰 집단이 유리한 것은 숨길 게 아니라 보고할 성질이기 때문임.
# 유리한 이유가 둘인데 작용하는 자리가 다름 —
#   ① 표본이 커서 분모가 작음 (9-4 의 N^(-1/3) 법칙)  → '보고 하한' 에 작용
#   ② δ 를 심으면 전체기준선까지 끌어내려, 전체 기준선을 쓰는 작은 경쟁자들을 눌러 버림.
#      큰 집단 자신은 집단별 보정을 받아 영향이 없음(같은 사건을 두 척도로 재는 비대칭).
#                                                   → '순위(찾음 하한)' 에만 작용
# 작용하는 자리가 달라 계산으로 갈라짐 — '찾음 하한 ÷ 보고 하한' 이 ②의 크기임.
# 그래서 거르는 대신 '비중' 칸을 같이 들고 가고, 크기가 한쪽에 몰리지 않게만 뽑음.
칸경계 = [0, 0.2, 0.7, 1.01]
칸이름 = ['많이 희석(r≤0.2)', '중간(0.2~0.7)', '거의 안 됨(0.7+)']
고를개수 = 5

# 9-5 를 안 돌렸어도 저장해 둔 파일이 있으면 그것으로 이어감
import os
if '조용쌍' not in dir() and os.path.exists(저장경로('쌍_조용')):
    조용쌍 = pd.read_csv(저장경로('쌍_조용'), encoding='utf-8-sig')
    print('{} 에서 {}개를 불러옴 (9-5 를 다시 안 돌려도 됨)'.format(저장경로('쌍_조용'), len(조용쌍)))

평가행수 = int((df['구간'].values == '평가').sum())
표 = 조용쌍.copy()
표['비중'] = 표['평가행'] / 평가행수            # 평가 기간 전체에서 차지하는 몫
표['희석칸'] = pd.cut(표['희석r'], 칸경계, labels=칸이름)
print('[칸마다 몇 개 있나]')
print(표['희석칸'].value_counts().sort_index().to_string())

뽑은줄 = []
축쓴횟수 = {}
for 칸 in 칸이름:
    후보 = 표[표['희석칸'] == 칸].copy()
    if len(후보) == 0:
        continue
    # 칸 안에서 크기를 5등분하고 조각마다 가장 조용한 것 하나씩 — 크기가 쏠리지 않게
    쪼갤수 = min(고를개수, len(후보))
    후보['크기조각'] = pd.qcut(후보['평가행'], 쪼갤수, labels=False, duplicates='drop')
    후보['조용함'] = 후보['t'].abs()
    for 조각 in sorted(후보['크기조각'].unique()):
        몫 = 후보[후보['크기조각'] == 조각].sort_values('조용함')
        for i in range(len(몫)):
            줄 = 몫.iloc[i]
            # 한 축이 표를 뒤덮지 않게 축마다 최대 3번까지만
            if 축쓴횟수.get(줄['축A'], 0) >= 3 or 축쓴횟수.get(줄['축B'], 0) >= 3:
                continue
            축쓴횟수[줄['축A']] = 축쓴횟수.get(줄['축A'], 0) + 1
            축쓴횟수[줄['축B']] = 축쓴횟수.get(줄['축B'], 0) + 1
            뽑은줄.append(줄)
            break

정답쌍 = pd.DataFrame(뽑은줄).reset_index(drop=True)
정답쌍['예측하한'] = 정답쌍[['보고δ예측', '도달δ예측']].max(axis=1)
정답쌍['무엇에막히나'] = np.where(정답쌍['도달δ예측'] > 정답쌍['보고δ예측'],
                            '도달(빔)', '보고(판정선)')
# 10/12 정확식으로 다시 센 결과 — 2일 창은 15개 중 0개, 14일은 1개만 도달에 막힘.
# 2일 창은 1단계 빔 문턱이 −0.823 으로 0 에 가까워 부모가 이미 빔 안인 일이 흔함.

print('\n[고른 정답 집단 {}개]'.format(len(정답쌍)))
print(정답쌍[['A', 'B', '희석칸', '평가행', '비중', '유효인원', 't', '희석r',
            '보고δ예측', '도달δ예측', '예측하한', '무엇에막히나']].round(3).to_string())

print('\n[칸별 요약]')
print(정답쌍.groupby('희석칸', observed=True).agg(
    개수=('A', 'count'), r중앙=('희석r', 'median'),
    평가행최소=('평가행', 'min'), 평가행최대=('평가행', 'max'),
    비중중앙=('비중', 'median'), 예측하한중앙=('예측하한', 'median'),
    빔에막힘=('무엇에막히나', lambda x: (x == '도달(빔)').sum())).round(3).to_string())
정답쌍.to_csv(저장경로('정답쌍'), index=False, encoding='utf-8-sig')
print('\n저장: ' + 저장경로('정답쌍') + '   (9-7 이 커널을 새로 띄워도 이어갈 수 있게)')

print('\n비중이 큰 집단은 두 가지로 유리함 — 분모가 작고(N^-1/3), 전체기준선을 끌어내려')
print('작은 경쟁자를 누름. 결과를 읽을 때 이 둘을 섞어 보지 말 것.')

[칸마다 몇 개 있나]
희석칸
많이 희석(r≤0.2)    150
중간(0.2~0.7)     282
거의 안 됨(0.7+)    238

[고른 정답 집단 15개]
                               A                        B           희석칸     평가행     비중      유효인원      t    희석r   보고δ예측  도달δ예측    예측하한   무엇에막히나
0           is_auto_renew = (결측)     days_since_last 3-6일  많이 희석(r≤0.2)     580  0.000     411.0 -0.011  0.031   7.746  0.000   7.746  보고(판정선)
1             days_since_last 결측  payment_plan_days 30~90  많이 희석(r≤0.2)    1752  0.001    1025.0 -0.063  0.083   5.091  5.563   5.563    도달(빔)
2              days_to_expire 결측    mins_per_day ≤82.2579  많이 희석(r≤0.2)    2130  0.002    1505.5 -0.003  0.113   5.123  0.046   5.123  보고(판정선)
3                      age = 40+   payment_method_id = 38  많이 희석(r≤0.2)    5926  0.005    3528.0  0.095  0.079   2.770  0.278   2.770  보고(판정선)
4         days_since_signup ≤358       days_to_expire >47  많이 희석(r≤0.2)   34477  0.029   20113.0 -0.035  0.157   1.054  0.236   1.054  보고(판정선)
5         days_since_signup ≤746   payment_method

### 찾음 하한 재기 (9-7)

주 지표가 Hit 이 되었으므로 **찾음 하한**이 본 숫자다 — 빔 서치 결과 **상위 3위 안에 정답이
들어오는 가장 작은 δ**. 순위를 알려면 경쟁자 6,000개를 다 매겨야 하므로 **빔 서치가 필요하다**
(판당 약 70~100초). 보고 하한처럼 싸게 못 잰다.

그래서 δ 격자를 훑지 않고 **이분 탐색**으로 좁힌다. 상한에서 한 판 돌려 보고,
되면 절반으로 내리고 안 되면 절반을 올리는 식이다.

**상한은 20%p 로 고정한다 (10/12).** 처음에는 `max(예측 × 1.4, 2.0)` 으로 좁게 잡았는데,
그러면 못 찾았을 때 **「예측의 1.4배까지는 못 찾았다」 밖에 말할 수 없다.** 실제로 2일 창에서
둘이 그렇게 걸렸고, 20%p 로 다시 열어 보니 **둘 다 찾혔다**(4.06 / 12.81%p). 그때는 못 찾은
것만 다시 보는 셀(9-8)을 따로 뒀는데, 이제 처음부터 20 으로 재므로 **한 셀로 합쳤다.**

| 범위 | 판수 | 눈금 |
|---|---|---|
| 0~20 | 4 | 1.25%p ← 너무 거침 |
| **0~20** | **6** | **0.31%p** |

| | 집단 수 | 판 수 | 시간 |
|---|---|---|---|
| 조건 2개 (9-6) | 15 | 105 | **약 100분** |

조건 1개짜리는 8절에서 이미 50판씩 돌렸으므로(소·중·대) 여기서는 다시 재지 않는다.
중간에 멈춰도 `results/` 에 거기까지 남는다. 함수(`한판` · `찾음하한`)는 바로 아래 셀에 있고,
**9-11(조건 3개)도 그 함수를 쓰므로** 루프와 따로 둔다.

In [7]:
# ── 9-7 준비. 한 판 돌리기 · 이분 탐색 함수 (정의만, 바로 끝남) ───────
# 아래 9-7(조건 2개, 약 100분)과 9-11(조건 3개, 약 80분)이 둘 다 이 함수를 씀.
# 그래서 루프와 떼어 놓았음 — 한쪽만 돌릴 때 다른 쪽 루프를 지나치지 않아도 되게 하려고임
def 한판(행, 정답행, 델타, seed):
    """δ 를 심고 빔 서치를 돌려 (순위, 최고겹침) 을 돌려줌"""
    편차 = 주입(행, 델타, seed)
    if 편차 is None:
        return 0, 0.0
    표, 가설목록, 최저t = 빔서치(편차=편차, 최소보정행=보정행, 출력=False)
    순위, 최고겹침 = 0, 0.0
    for i in range(len(가설목록)):
        그행 = 가설목록[i]['행번호']
        같이 = len(np.intersect1d(그행, 정답행, True))
        겹침 = 같이 / (len(그행) + len(정답행) - 같이)
        최고겹침 = max(최고겹침, 겹침)
        if 겹침 >= 0.5 and 순위 == 0:
            순위 = i + 1
    return 순위, 최고겹침


def 찾음하한(조건들, 위, 아래=0.0, 판수=4, seed=20261009):
    """정답이 상위 3위 안에 드는 가장 작은 δ 를 이분 탐색으로 좁힘.
       돌려주는 것 — (하한 추정, 판별 기록). 상한에서도 못 찾으면 하한은 None.

    ⚠️ 상한을 먼저 한 판 돌려 봄. 이게 없으면 "상한에서 찾았다" 와
       "상한까지 올려도 못 찾았다" 가 **같은 숫자로 찍힘** (둘 다 위 를 돌려주므로).
       10/11 에 전체 평가의 76% 짜리 집단에서 실제로 겪은 문제임.
    """
    행 = 집단행(조건들)
    기록 = []

    def 한번(델타, n):
        순위, 겹침 = 한판(행, 행, 델타, seed + n)
        찾음 = (1 <= 순위 <= 3) and (겹침 >= 0.5)
        기록.append({'δ': 델타, '순위': 순위, '겹침': 겹침, '찾음': 찾음})
        return 찾음

    if not 한번(위, 0):                  # 상한에서도 못 찾으면 이 범위 안에 답이 없음
        return None, pd.DataFrame(기록)

    for n in range(판수):
        가운데 = (아래 + 위) / 2
        if 한번(가운데, n + 1):
            위 = 가운데                   # 더 작은 δ 로도 되는지 봄
        else:
            아래 = 가운데                 # 더 키워야 함
    return 위, pd.DataFrame(기록)

In [15]:
# ── 9-7. 찾음 하한 — 이분 탐색 (15개 × 7판, 약 100분) ──────────────────
# 10/12 에 9-8(못 찾은 것만 20%p 로 다시)과 합침. 상한을 처음부터 20 으로 두므로
# 「못 찾음」이 「예측 × 1.4 까지만 봤음」을 뜻하지 않게 됨. 판수 6 → 눈금 0.31%p.
# 함수(한판 · 찾음하한)는 위 셀에 있음
import os
os.makedirs('results', exist_ok=True)

# 9-6 을 안 돌렸어도 저장해 둔 파일이 있으면 그것으로 이어감
if '정답쌍' not in dir() and os.path.exists(저장경로('정답쌍')):
    정답쌍 = pd.read_csv(저장경로('정답쌍'), encoding='utf-8-sig')
    print('{} 에서 {}개를 불러옴 (9-6 을 다시 안 돌려도 됨)'.format(저장경로('정답쌍'), len(정답쌍)))

# 옛 정답쌍 파일에는 비중·무엇에막히나 가 없을 수 있음 (9-6 에 그 칸이 생기기 전에 만든 것).
# 없으면 여기서 채워 **어느 쪽 파일로 이어가도 칸이 같게** 함
if '비중' not in 정답쌍.columns:
    정답쌍['비중'] = 정답쌍['평가행'] / int((df['구간'].values == '평가').sum())
if '무엇에막히나' not in 정답쌍.columns:
    정답쌍['무엇에막히나'] = np.where(정답쌍['도달δ예측'] > 정답쌍['보고δ예측'],
                                '도달(빔)', '보고(판정선)')

상한 = 20.0

# 끊긴 데서 이어감 — 저장해 둔 파일에 있는 집단은 건너뜀.
# 14일 창은 집단당 17~50분이라 처음부터 다시 재면 몇 시간을 버림
줄 = []
if os.path.exists(저장경로('찾음하한_2조건')):
    먼저 = pd.read_csv(저장경로('찾음하한_2조건'), encoding='utf-8-sig')
    if set(['A', 'B']).issubset(먼저.columns):
        # ⚠️ 그 파일이 **다른 정답셋**(옛 조건으로 뽑은 것) 결과일 수 있음.
        #    그대로 이어가면 옛 줄이 안 버려지고 새 줄이 덧붙어 15개가 29개가 됨
        #    (10/15 에 조건 3개에서 실제로 그랬음). 지금 정답셋에 있는 것만 남김
        지금 = set(zip(정답쌍['A'], 정답쌍['B']))
        줄 = [r for r in 먼저.to_dict('records') if (r['A'], r['B']) in 지금]
        버림 = len(먼저) - len(줄)
        print('{} 에서 {}개를 불러옴{} — 나머지만 돌림'.format(
            저장경로('찾음하한_2조건'), len(줄),
            ' (지금 정답셋에 없는 {}개는 버림)'.format(버림) if 버림 else ''))
끝난것 = {(r['A'], r['B']) for r in 줄}

시작 = time.time()
for i in range(len(정답쌍)):
    줄정답 = 정답쌍.iloc[i]
    if (줄정답['A'], 줄정답['B']) in 끝난것:
        print('{:>2}/{}  이미 끝남 — 건너뜀'.format(i + 1, len(정답쌍)))
        continue
    조건들 = [줄정답['A'], 줄정답['B']]
    하한, 기록 = 찾음하한(조건들, 위=상한, 판수=6)
    # 칸 이름·차례는 9-11(조건 3개)과 똑같이 맞춤. 10/12 에 두 파일 칸이 어긋나
    # 같은 이름이 다른 뜻이 된 적이 있어(도달δ예측 이 한쪽은 근사, 한쪽은 정확식)
    # 두 파일을 나란히 읽을 수 있게 통일함
    줄.append({'A': 줄정답['A'], 'B': 줄정답['B'], '희석칸': 줄정답['희석칸'],
              '평가행': 줄정답['평가행'], '비중': 줄정답['비중'],
              '희석r': 줄정답['희석r'],
              '보고δ예측': 줄정답['보고δ예측'], '도달δ예측': 줄정답['도달δ예측'],
              '예측하한': 줄정답['예측하한'], '무엇에막히나': 줄정답['무엇에막히나'],
              '찾음하한': 하한,
              '상한': 상한, '상한에서찾음': 하한 is not None,
              '최고겹침': 기록['겹침'].max()})
    print('{:>2}/{}  {:<28s} × {:<24s} r={:.3f}  예측 {:>6.2f} → 찾음 {:>8s}  ({:.0f}분)'.format(
        i + 1, len(정답쌍), 줄정답['A'][:28], 줄정답['B'][:24], 줄정답['희석r'],
        줄정답['예측하한'],
        '{:.2f}'.format(하한) if 하한 is not None else '>{:.1f}'.format(상한),
        (time.time() - 시작) / 60))
    print(기록.round(3).to_string(index=False))      # 어느 δ 에서 되고 안 됐는지가 본 자료임
    # 집단 하나 끝날 때마다 저장 — 2시간짜리라 중간에 끊기면 전부 날아감
    pd.DataFrame(줄).to_csv(저장경로('찾음하한_2조건'), index=False, encoding='utf-8-sig')

찾음표 = pd.DataFrame(줄)
찾음표.to_csv(저장경로('찾음하한_2조건'), index=False, encoding='utf-8-sig')
print('\n저장: {}   ({:.0f}분)'.format(저장경로('찾음하한_2조건'), (time.time() - 시작) / 60))

찾음표['비율'] = 찾음표['찾음하한'] / 찾음표['예측하한']
print('\n[희석 정도별]')
print(찾음표.groupby('희석칸', observed=True).agg(
    개수=('A', 'count'), 찾음=('상한에서찾음', 'sum'), r중앙=('희석r', 'median'),
    예측하한중앙=('예측하한', 'median'), 찾음하한중앙=('찾음하한', 'median'),
    비율중앙=('비율', 'median')).round(3).to_string())
print('\n찾은 집단 {}개 / {}개'.format(int(찾음표['상한에서찾음'].sum()), len(찾음표)))
print('찾음 ÷ 예측 — 희석이 심한 칸에서 이 비율이 커지면 도달 관문이 실제로 작동하는 것임')
print('(2일 창 10/12 결과: 15/15 찾음, 칸별 중앙 0.935 / 0.875 / 1.169)')

results/찾음하한_2조건_2일.csv 에서 7개를 불러옴 — 나머지만 돌림
 1/15  이미 끝남 — 건너뜀
 2/15  이미 끝남 — 건너뜀
 3/15  이미 끝남 — 건너뜀
 4/15  이미 끝남 — 건너뜀
 5/15  이미 끝남 — 건너뜀
 6/15  이미 끝남 — 건너뜀
 7/15  이미 끝남 — 건너뜀
 8/15  channel = 7                  × days_since_last >2       r=0.452  예측   1.67 → 찾음     1.88  (11분)
     δ  순위    겹침    찾음
20.000   1 0.753  True
10.000   1 0.753  True
 5.000   1 0.753  True
 2.500   1 0.834  True
 1.250   0 0.320 False
 1.875   2 0.834  True
 1.562   6 0.638 False
 9/15  pct_unique 0.838955~0.901478 × weekend_ratio >0.291667  r=0.396  예측   1.19 → 찾음     1.88  (28분)
     δ  순위    겹침    찾음
20.000   2 0.864  True
10.000   2 0.684  True
 5.000   2 0.684  True
 2.500   3 0.684  True
 1.250   9 0.684 False
 1.875   2 0.684  True
 1.562   4 0.684 False
10/15  days_since_signup ≤1343      × trend >-1                r=0.642  예측   0.40 → 찾음     0.62  (63분)
     δ  순위    겹침    찾음
20.000   1 0.848  True
10.000   1 0.848  True
 5.000   1 0.848  True
 2.500   1 0.848  True
 1.250   1 0.848  True
 0.625 

### 조건 3개 — 관문이 두 번 (9-9 ~ 9-11)

빔 서치는 조건을 하나씩 붙이므로 조건 3개짜리 집단은 **두 번** 살아남아야 만들어진다.

```
264개 단일 조건 ──1단계 상위 20──▶ 쌍 ──2단계 상위 20──▶ 셋
   (조부모)                    (부모)                (정답 집단)
```

한 길이 통하려면 **조부모가 1단계에 들고, 그 조부모를 품은 쌍이 2단계에도 들어야** 한다.
그래서 길의 값은 둘 중 **큰** 쪽이고, 집단의 도달δ 는 길 `(조부모 3) × (그를 품은 쌍 2) = 6`개 중
**가장 싼** 쪽이다.

In [16]:
# ── 9-9. 조건 3개 후보를 무작위로 채점 (약 3분) ──────────────────────────
# 조용한 단일 조건만 재료로 씀 — 조부모가 원래 떠 있으면 시험이 불공정함 (9-5 와 같은 이유)
# 2단계 문턱은 저장해 둔 값이 없어 무주입 한 판으로 뽑음 (5절 `빔서치()` 가 `단계문턱` 에 남김)
_, _, 판정선 = 빔서치(최소보정행=보정행, 출력=True)
문턱1, 문턱2 = 단계문턱[0], 단계문턱[1]
print('\n판정선 {:.4f} / 단계 문턱 {}'.format(판정선, [round(x, 4) for x in 단계문턱]))
print('2단계 문턱은 판정선의 {:.0%} — 부모 쌍이 「거의 보고 가능할 만큼」 떠야 자식이 생김'.format(
    문턱2 / 판정선))

학습개수 = int((df['구간'].values == '학습').sum())
평가전체 = int((df['구간'].values == '평가').sum())
무주입 = pd.read_csv(저장경로('무주입_조건'), encoding='utf-8-sig')
# ⚠️ 14일 파일은 **조건 중복 26개를 바로잡기 전**에 만든 것이라 같은 이름이 22종(44줄) 있음.
#    두 줄의 값이 글자까지 똑같으므로 어느 쪽을 남겨도 됨. 안 빼면 두 군데서 탈이 남 —
#    ① 여기서 중복 조건이 **두 배 확률로** 뽑혀 후보가 공정하지 않고
#    ② 9-10 의 `단일.loc[이름]` 이 두 줄을 돌려줘 max() 가 터짐
#       (ValueError: The truth value of a Series is ambiguous)
전체줄 = len(무주입)
무주입 = 무주입.drop_duplicates('이름').reset_index(drop=True)
if len(무주입) != 전체줄:
    print('이름 중복 {}줄 뺌 ({} → {}) — 10/11 조건 중복을 고치기 전 파일임'.format(
        전체줄 - len(무주입), 전체줄, len(무주입)))
조용 = 무주입[무주입['t'].abs() < 1].reset_index(drop=True)
print('\n조용한 단일 조건 {}개 (축 {}종) 로 셋을 만듦'.format(len(조용), 조용['축'].nunique()))

뽑을수 = 300
난수 = np.random.default_rng(20261012)
본것, 고른셋 = set(), []
while len(고른셋) < 뽑을수:
    i, j, k = 난수.choice(len(조용), 3, replace=False)
    if len({조용['축'].iloc[i], 조용['축'].iloc[j], 조용['축'].iloc[k]}) < 3:
        continue                              # 같은 축은 빔이 두 번 안 붙임
    열쇠 = frozenset([i, j, k])
    if 열쇠 in 본것:
        continue
    본것.add(열쇠)
    고른셋.append((int(i), int(j), int(k)))


def 평가수(행):
    return int((행 >= 학습개수).sum())


시작 = time.time()
줄 = []
for n, (i, j, k) in enumerate(고른셋):
    이름 = [조용['이름'].iloc[i], 조용['이름'].iloc[j], 조용['이름'].iloc[k]]
    점 = 집단점수(집단행(이름), 최소보정행=보정행, 출력=False)
    if 점 is None:                            # 유효 인원 100명 미만 — 조건 3개는 이게 흔함
        continue
    쌍행수 = [평가수(집단행([x, y])) for x, y in
            [(이름[0], 이름[1]), (이름[0], 이름[2]), (이름[1], 이름[2])]]
    분모 = (점['평가오차'] ** 2 + 점['기준선오차'] ** 2) ** 0.5
    줄.append({'A': 이름[0], 'B': 이름[1], 'C': 이름[2],
              '축A': 조용['축'].iloc[i], '축B': 조용['축'].iloc[j], '축C': 조용['축'].iloc[k],
              '평가행': 점['평가행'], '유효인원': 점['유효인원'], 't': 점['t'],
              '보정후': 점['보정후'], '보정': 점['보정'], '분모': 분모,
              # 희석 — 자식이 **직전 부모(가장 작은 쌍)** 에서 차지하는 몫. 쌍 연구와 같은 뜻임
              '희석r': 점['평가행'] / min(쌍행수), '작은쌍평가행': min(쌍행수),
              '보고δ예측': 점['보정후'] + abs(판정선) * 분모})
    if (n + 1) % 50 == 0:
        print('  {}/{} — 살아남음 {}개 ({:.0f}초)'.format(
            n + 1, len(고른셋), len(줄), time.time() - 시작))

셋 = pd.DataFrame(줄)
조용셋 = 셋[셋['t'].abs() < 1].reset_index(drop=True)
print('\n채점 {}개 (인원 미달 {}개 빠짐) / 자식도 조용한 것 {}개   ({:.0f}분)'.format(
    len(셋), len(고른셋) - len(셋), len(조용셋), (time.time() - 시작) / 60))
print('\n[희석 r 퍼짐] — 조건 3개는 작은 조합이 인원 미달로 죽어 r 이 높은 쪽에 쏠림')
print(조용셋['희석r'].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
print('\n[평가행 퍼짐]')
print(조용셋['평가행'].describe(percentiles=[.25, .5, .75]).round(0).to_string())

import os
os.makedirs('results', exist_ok=True)
조용셋.to_csv(저장경로('셋_조용'), index=False, encoding='utf-8-sig')
print('\n저장: ' + 저장경로('셋_조용'))

1단계 — 후보 225개 / 최저 t -1.43   (14초)
2단계 — 후보 2,826개 / 최저 t -2.97   (41초)
3단계 — 후보 2,126개 / 최저 t -3.64   (51초)
후보 5,177개 / 최저 t -3.64 / 전체 기준선 +1.082%p   (60초)
                                                                            가설  조건     인원  보정후(%p)  ±기준선   기울기     t
0                            trend >3 AND payment_plan_days 결측 AND channel ≠ 9   3    229   -11.45 +0.00 +0.00 -3.64
1     days_since_last 7-13일 AND weekend_ratio >0.291667 AND pct_full >0.791633   3  3,151    -4.52 +0.94 +0.75 -3.56
2             act_days 21-28일 AND payment_method_id = 41 AND days_to_expire ≤9   3 31,696    -0.81 +0.14 +0.01 -3.55
3  days_since_last 7-13일 AND weekend_ratio >0.291667 AND mins_per_day >56.1611   3  3,951    -3.46 +0.75 +0.72 -3.24
4       payment_method_id = 15 AND days_to_expire 26~47 AND pct_full ≤0.791633   3    200    -8.69 +0.00 +0.00 -3.20
5                   age = 10s AND mins_per_day 56.1611~82.2579 AND channel ≠ 4   3  5,417    -1.92 +0.34 +0.23 -3.15
6               plan_li

In [17]:
# ── 9-10. 희석 칸마다 5개씩 고르고 관문 두 번을 예측 (약 1분) ───────────
# 고르는 방식은 9-6 과 같음 — 크기로 거르지 않고, 칸 안에서 크기가 쏠리지 않게만 뽑음
칸경계 = [0, 0.2, 0.7, 1.01]
칸이름 = ['많이 희석(r≤0.2)', '중간(0.2~0.7)', '거의 안 됨(0.7+)']
고를개수 = 5

import os
if '조용셋' not in dir() and os.path.exists(저장경로('셋_조용')):
    조용셋 = pd.read_csv(저장경로('셋_조용'), encoding='utf-8-sig')
    print('{} 에서 {}개를 불러옴 (9-9 를 다시 안 돌려도 됨)'.format(저장경로('셋_조용'), len(조용셋)))

표 = 조용셋.copy()
표['비중'] = 표['평가행'] / int((df['구간'].values == '평가').sum())
표['희석칸'] = pd.cut(표['희석r'], 칸경계, labels=칸이름)
print('[칸마다 몇 개 있나]')
print(표['희석칸'].value_counts().sort_index().to_string())

뽑은줄, 축쓴횟수 = [], {}
for 칸 in 칸이름:
    후보 = 표[표['희석칸'] == 칸].copy()
    if len(후보) == 0:
        continue
    쪼갤수 = min(고를개수, len(후보))
    후보['크기조각'] = pd.qcut(후보['평가행'], 쪼갤수, labels=False, duplicates='drop')
    후보['조용함'] = 후보['t'].abs()
    for 조각 in sorted(후보['크기조각'].unique()):
        몫 = 후보[후보['크기조각'] == 조각].sort_values('조용함')
        for 엔 in range(len(몫)):
            줄하나 = 몫.iloc[엔]
            축들 = [줄하나['축A'], 줄하나['축B'], 줄하나['축C']]
            if any(축쓴횟수.get(x, 0) >= 3 for x in 축들):   # 한 축이 표를 뒤덮지 않게
                continue
            for x in 축들:
                축쓴횟수[x] = 축쓴횟수.get(x, 0) + 1
            뽑은줄.append(줄하나)
            break

정답셋 = pd.DataFrame(뽑은줄).reset_index(drop=True)

# ── 관문 두 번. 길 = (조부모 s, s 를 품은 쌍 p) 6가지, 길의 값은 max(δ_s, δ_p) ──
# 쓰는 사실은 하나뿐이고 그건 등식임 — 자식 평가행의 δ% 를 뒤집으면 조상의 평균 편차가
# 정확히 δ × r 내려감 (r = 자식 평가행 ÷ 조상 평가행). 10/12 에 `√r` 근사를 버렸으므로
# 조부모·부모를 **직접 채점**해 씀
# 이름으로 한 줄씩 꺼내 쓰므로 **중복이 있으면 안 됨** (9-9 의 주석 참고).
# 9-9 를 안 거치고 이 셀부터 돌리는 경우도 있어 여기서도 한 번 더 막음
단일 = (조용 if '조용' in dir()
      else pd.read_csv(저장경로('무주입_조건'), encoding='utf-8-sig')
      ).drop_duplicates('이름').set_index('이름')
assert 단일.index.is_unique, '단일 조건 이름이 중복임 — drop_duplicates 가 안 먹었음'
도달들, 길설명 = [], []
for _, r in 정답셋.iterrows():
    이름 = [r['A'], r['B'], r['C']]
    쌍점 = {}
    for p in [(이름[0], 이름[1]), (이름[0], 이름[2]), (이름[1], 이름[2])]:
        점 = 집단점수(집단행(list(p)), 최소보정행=보정행, 출력=False)
        쌍점[p] = None if 점 is None else {
            '평가행': 점['평가행'], '보정후': 점['보정후'],
            '분모': (점['평가오차'] ** 2 + 점['기준선오차'] ** 2) ** 0.5}
    최선, 최선설명 = None, ''
    for s in 이름:
        조 = 단일.loc[s]
        델타_s = (조['보정후'] + abs(문턱1) * 조['분모']) / (r['평가행'] / 조['평가행'])
        for p, 점 in 쌍점.items():
            if s not in p or 점 is None:
                continue
            델타_p = (점['보정후'] + abs(문턱2) * 점['분모']) / (r['평가행'] / 점['평가행'])
            이길 = max(델타_s, 델타_p, 0.0)      # 음수면 심기 전에 이미 빔 안
            if 최선 is None or 이길 < 최선:
                최선, 최선설명 = 이길, '{} → +{}'.format(
                    s[:22], [x for x in p if x != s][0][:22])
    도달들.append(최선 if 최선 is not None else np.nan)
    길설명.append(최선설명)

정답셋['도달δ예측'] = 도달들
정답셋['싼길'] = 길설명
정답셋['예측하한'] = 정답셋[['보고δ예측', '도달δ예측']].max(axis=1)
정답셋['무엇에막히나'] = np.where(정답셋['도달δ예측'] > 정답셋['보고δ예측'], '도달(빔)', '보고(판정선)')
정답셋['판정선'], 정답셋['문턱1'], 정답셋['문턱2'] = 판정선, 문턱1, 문턱2

print('\n[고른 정답 집단 {}개 — 창 {}]'.format(len(정답셋), 창이름))
print(정답셋[['A', 'B', 'C', '희석칸', '평가행', '비중', 't', '희석r',
            '보고δ예측', '도달δ예측', '예측하한', '무엇에막히나']].round(3).to_string())
print('\n[칸별 요약]')
print(정답셋.groupby('희석칸', observed=True).agg(
    개수=('A', 'count'), r중앙=('희석r', 'median'),
    평가행중앙=('평가행', 'median'), 비중중앙=('비중', 'median'),
    보고중앙=('보고δ예측', 'median'), 도달중앙=('도달δ예측', 'median'),
    예측하한중앙=('예측하한', 'median'),
    빔에막힘=('무엇에막히나', lambda x: (x == '도달(빔)').sum())).round(3).to_string())
print('\n도달에 막히는 집단 {}개 / {}개   (조건 2개 2일 창은 15개 중 0개였음)'.format(
    (정답셋['무엇에막히나'] == '도달(빔)').sum(), len(정답셋)))

정답셋.to_csv(저장경로('정답셋'), index=False, encoding='utf-8-sig')
print('저장: ' + 저장경로('정답셋'))

[칸마다 몇 개 있나]
희석칸
많이 희석(r≤0.2)     6
중간(0.2~0.7)     49
거의 안 됨(0.7+)    49

[고른 정답 집단 15개 — 창 2일]
                                  A                            B                             C           희석칸     평가행     비중      t    희석r   보고δ예측   도달δ예측    예측하한   무엇에막히나
0                         trend 1~3       payment_method_id = 22             profile = missing  많이 희석(r≤0.2)     437  0.000 -0.099  0.196   8.184  13.409  13.409    도달(빔)
1            payment_method_id = 38         pct_unique ≤0.640086                  act_days ≤10  많이 희석(r≤0.2)    3017  0.003  0.910  0.126   4.909   6.463   6.463    도달(빔)
2   weekend_ratio 0.235294~0.272727                  channel = 4         mins_per_day >176.145  많이 희석(r≤0.2)    3720  0.003 -0.471  0.196   2.516   4.968   4.968    도달(빔)
3         days_since_signup 358~746              gender = female                act_days 4-10일  많이 희석(r≤0.2)    4250  0.004  0.456  0.158   4.778  10.119  10.119    도달(빔)
4              pct_unique ≤0.640086  weekend_rat

In [8]:
# ── 9-11. 조건 3개 찾음 하한 — 이분 탐색 (15개 × 7판, 약 80분) ──────────
# 상한은 max(20, 예측 × 1.3). 9-8 에서 「예측 × 1.4 뿐이면 못 찾은 집단에 대해
# 아무 말도 못 한다」를 겪었으므로 바닥을 20 으로 두고, 예측이 20 을 넘으면 그 위까지 봄.
# 함수(한판 · 찾음하한)는 9-7 위의 정의 셀에 있음
import os
if '정답셋' not in dir() and os.path.exists(저장경로('정답셋')):
    정답셋 = pd.read_csv(저장경로('정답셋'), encoding='utf-8-sig')
    print('{} 에서 {}개를 불러옴'.format(저장경로('정답셋'), len(정답셋)))
print('판정선 {:.4f} / 문턱1 {:.4f} / 문턱2 {:.4f}'.format(
    정답셋['판정선'].iloc[0], 정답셋['문턱1'].iloc[0], 정답셋['문턱2'].iloc[0]))

# 끊긴 데서 이어감 — 저장해 둔 파일에 있는 집단은 건너뜀 (9-7 과 같은 이유)
줄 = []
if os.path.exists(저장경로('찾음하한_3조건')):
    먼저 = pd.read_csv(저장경로('찾음하한_3조건'), encoding='utf-8-sig')
    if set(['A', 'B', 'C']).issubset(먼저.columns):
        # ⚠️ 9-7 과 같은 이유 — 다른 정답셋 결과면 버림 (10/15 에 29줄이 됐었음)
        지금 = set(zip(정답셋['A'], 정답셋['B'], 정답셋['C']))
        줄 = [r for r in 먼저.to_dict('records')
             if (r['A'], r['B'], r['C']) in 지금]
        버림 = len(먼저) - len(줄)
        print('{} 에서 {}개를 불러옴{} — 나머지만 돌림'.format(
            저장경로('찾음하한_3조건'), len(줄),
            ' (지금 정답셋에 없는 {}개는 버림)'.format(버림) if 버림 else ''))
끝난것 = {(r['A'], r['B'], r['C']) for r in 줄}

시작 = time.time()
for n in range(len(정답셋)):
    r = 정답셋.iloc[n]
    if (r['A'], r['B'], r['C']) in 끝난것:
        print('{:>2}/{}  이미 끝남 — 건너뜀'.format(n + 1, len(정답셋)))
        continue
    # 상한 — 넉넉히 올리되(× 1.3 → × 1.5) **그 집단이 실제로 뒤집을 수 있는 천장**으로 자름.
    # 주입은 재방문 1 을 0 으로 바꾸는 것이라 돌아온 사람보다 많이 뒤집을 수 없음.
    # 10/12 에 꼭대기가 105%p 로 잡혀 **한 판 만에 「못 찾음」**이 된 집단이 있었음
    # (상한에서 실패하면 찾음하한() 이 즉시 None 이라 이분 탐색이 시작도 안 함)
    행3 = 집단행([r['A'], r['B'], r['C']])
    평가행3 = 행3[행3 >= int((df['구간'].values == '학습').sum())]
    천장 = 100.0 * float((df['d1'].values[평가행3] > 0.5).mean())
    꼭대기 = min(max(20.0, float(r['예측하한']) * 1.5), 천장)
    하한, 기록 = 찾음하한([r['A'], r['B'], r['C']], 위=꼭대기, 판수=6)
    줄.append({'A': r['A'], 'B': r['B'], 'C': r['C'], '희석칸': r['희석칸'],
              '평가행': r['평가행'], '비중': r['비중'], '희석r': r['희석r'],
              '보고δ예측': r['보고δ예측'], '도달δ예측': r['도달δ예측'],
              '예측하한': r['예측하한'], '무엇에막히나': r['무엇에막히나'],
              '찾음하한': 하한, '상한': 꼭대기, '상한에서찾음': 하한 is not None,
              '최고겹침': 기록['겹침'].max()})
    print('{:>2}/{}  {:<24.24s} × {:<20.20s} × {:<20.20s} r={:.3f} 예측 {:>6.2f} → 찾음 {:>8s} ({:.0f}분)'.format(
        n + 1, len(정답셋), r['A'], r['B'], r['C'], r['희석r'], r['예측하한'],
        '{:.2f}'.format(하한) if 하한 is not None else '>{:.1f}'.format(꼭대기),
        (time.time() - 시작) / 60))
    print(기록.round(3).to_string(index=False))
    # 집단 하나 끝날 때마다 저장 — 2시간짜리라 중간에 끊기면 전부 날아감
    pd.DataFrame(줄).to_csv(저장경로('찾음하한_3조건'), index=False, encoding='utf-8-sig')

찾음3 = pd.DataFrame(줄)
찾음3.to_csv(저장경로('찾음하한_3조건'), index=False, encoding='utf-8-sig')
찾음3['비율'] = 찾음3['찾음하한'] / 찾음3['예측하한']

print('\n[결과]')
print(찾음3[['A', 'B', 'C', '희석칸', '평가행', '희석r', '보고δ예측', '도달δ예측',
           '예측하한', '찾음하한', '상한', '최고겹침']].round(3).to_string())
print('\n[칸별 요약]')
print(찾음3.groupby('희석칸', observed=True).agg(
    개수=('A', 'count'), 찾음=('상한에서찾음', 'sum'), r중앙=('희석r', 'median'),
    예측중앙=('예측하한', 'median'), 찾음중앙=('찾음하한', 'median'),
    비율중앙=('비율', 'median')).round(3).to_string())
print('\n찾은 집단 {}개 / {}개   (조건 2개 2일 창은 15/15 였음)'.format(
    int(찾음3['상한에서찾음'].sum()), len(찾음3)))
print('찾음÷예측 중앙값 {:.3f} — 1 에 가까우면 예측식이 맞는 것임'.format(찾음3['비율'].median()))
print('\n저장: ' + 저장경로('찾음하한_3조건'))

results/정답셋_2일.csv 에서 15개를 불러옴
판정선 -3.6360 / 문턱1 -0.8226 / 문턱2 -2.4173
results/찾음하한_3조건_2일.csv 에서 14개를 불러옴 — 나머지만 돌림
 1/15  이미 끝남 — 건너뜀
 2/15  이미 끝남 — 건너뜀
 3/15  이미 끝남 — 건너뜀
 4/15  이미 끝남 — 건너뜀
 5/15  이미 끝남 — 건너뜀
 6/15  이미 끝남 — 건너뜀
 7/15  이미 끝남 — 건너뜀
 8/15  이미 끝남 — 건너뜀
 9/15  이미 끝남 — 건너뜀
10/15  이미 끝남 — 건너뜀
11/15  이미 끝남 — 건너뜀
12/15  이미 끝남 — 건너뜀
13/15  이미 끝남 — 건너뜀
14/15  이미 끝남 — 건너뜀
15/15  days_since_last ≤0       × pct_full >0.698507   × days_since_signup ≤2 r=0.780 예측   0.35 → 찾음     0.62 (32분)
     δ  순위    겹침    찾음
20.000   1 1.000  True
10.000   1 1.000  True
 5.000   1 1.000  True
 2.500   1 1.000  True
 1.250   1 1.000  True
 0.625   2 1.000  True
 0.312   0 0.328 False

[결과]
                                  A                            B                             C           희석칸     평가행    희석r   보고δ예측   도달δ예측    예측하한    찾음하한      상한   최고겹침
0                         trend 1~3       payment_method_id = 22             profile = missing  많이 희석(r≤0.2)     437  0.196   8.184  13.409 

## 10. 두 창 비교 (2일 vs 14일)

평가 창은 **2일이 주 분석, 14일이 보조**다 (10/11 결정). 창을 바꾸는 것은 **2절 한 줄**이고,
결과 파일은 창 이름으로 따로 저장되므로 서로 덮어쓰지 않는다.

```
창이름, 평가끝, 보정행 = '2일',  16, 1400      ← 주 분석
창이름, 평가끝, 보정행 = '14일', 28, 10000     ← 보조
```

왜 2일인가 — 14일 창은 **하락이 2주 내내 지속된다고 가정**하는 셈이라, 사건이 사흘 만에
복구되면 평균이 δ 의 21% 로 희석돼 못 찾는다. 연구 범위가 「급락 진단」인데 설계가
지속되는 하락만 잡는 어긋남이 있었다.

실측으로도 2일이 전 구간에서 낫다. 이유는 분모가 아니라 **판정선** 이다 —
외삽 거리가 14.5일에서 8.5일로 줄어 **거짓 신호 자체가 약해지기 때문**이다.

아래 셀은 두 창의 결과 파일을 읽어 맞대 본다. 두 창을 모두 돌린 뒤에 실행할 것.

In [51]:
# ── 10-1. 두 창의 보고 하한을 맞대 봄 ──────────────────────────────────
# ⚠️ 조건 이름이 중복인 것이 있었으므로(10/11 수정) 혹시 남아 있어도 걸러 내고,
#    보고 하한이 0 이하인 집단은 log 가 nan 이 되므로 함께 거름
import os

둘 = {}
for 창 in ['14일', '2일']:
    경로 = 'results/무주입_조건_{}.csv'.format(창)
    if not os.path.exists(경로):
        print('(없음) ' + 경로 + ' — 그 창으로 2절 → 9-1 → 9-3 을 돌릴 것')
        continue
    둘[창] = pd.read_csv(경로, encoding='utf-8-sig').drop_duplicates('이름')
    print('{:>4} 창 — 조건 {}개'.format(창, len(둘[창])))

if len(둘) == 2:
    맞춤 = 둘['14일'][['이름', '평가행', '유효인원', '보고하한']].merge(
        둘['2일'][['이름', '유효인원', '보고하한']], on='이름',
        suffixes=('_14일', '_2일')).dropna()
    맞춤 = 맞춤[(맞춤['보고하한_14일'] > 0) & (맞춤['보고하한_2일'] > 0)]
    맞춤['배수'] = 맞춤['보고하한_2일'] / 맞춤['보고하한_14일']
    print('두 창에 다 있고 보고 하한이 양수인 조건 {}개'.format(len(맞춤)))

    경계 = [0, 3000, 10000, 30000, 100000, 10 ** 9]
    이름표 = ['~3천', '3천~1만', '1만~3만', '3만~10만', '10만+']
    맞춤['크기'] = pd.cut(맞춤['평가행'], 경계, labels=이름표)   # 14일 창의 평가 행 기준
    print()
    print('[크기 구간별 보고 하한 — 배수가 1 보다 작으면 2일 창이 더 좋다는 뜻]')
    print(맞춤.groupby('크기', observed=True).agg(
        집단=('이름', 'count'),
        하한_14일=('보고하한_14일', 'median'),
        하한_2일=('보고하한_2일', 'median'),
        배수=('배수', 'median')).round(3).to_string())

    print()
    for 창, d in 둘.items():
        쓸것 = d[d['보고하한'] > 0]
        기울기 = np.polyfit(np.log(쓸것['유효인원']), np.log(쓸것['보고하한']), 1)[0]
        print('{:>4} 창 — log(유효인원) 대 log(보고하한) 기울기 {:+.3f} / 무주입 t 평균 {:+.2f}'.format(
            창, 기울기, d['t'].mean()))

 14일 창 — 조건 264개
  2일 창 — 조건 260개
두 창에 다 있고 보고 하한이 양수인 조건 259개

[크기 구간별 보고 하한 — 배수가 1 보다 작으면 2일 창이 더 좋다는 뜻]
         집단  하한_14일   하한_2일     배수
크기                                
~3천       4  24.283  12.635  0.561
3천~1만     5  15.058   8.871  0.691
1만~3만     6   9.062   4.263  0.467
3만~10만   10   5.001   1.826  0.417
10만+    234   1.298   0.594  0.457

 14일 창 — log(유효인원) 대 log(보고하한) 기울기 -0.326 / 무주입 t 평균 -0.78
  2일 창 — log(유효인원) 대 log(보고하한) 기울기 -0.346 / 무주입 t 평균 +0.83
